# Powderday flux catalogs — quenched galaxies in the high-res 25 Mpc box

**Goal.** Multi-aperture, dusty vs dust-free photometric catalogs (fluxes **with errors**) for
**quenched** galaxies in SIMBA high-res **m25n512** (`cis25`) at **z ≈ 0.3, 0.6, 0.7, 1.0, 2.0**.

**Sample (per anchor snapshot).** `log10 M* > 10`, **passive** by the 0.2/τ criterion
(sSFR < 0.2/t_H at the anchor), and **> 20 gas particles** (plus the usual ≥ 20 star-particle floor).
The sample is split by **weak vs strong AGN feedback over the quench window**: the AGN–ISM coupling
strength `xcoup_hist` (jet-mode strength gated by gas-poorness, §8j physics) averaged between each
galaxy's **SFT and QT** (1/t and 0.2/t crossings from `find_quenching_times`), tercile split.

**Pipeline** (same skeleton as `test_powderday.ipynb`, selection machinery from
`quench_mode_vs_sigma_gas.ipynb`):

| Part | What | Where |
|---|---|---|
| 1 | anchors + gated `BUILD_MULTI_Z` / `BUILD_BH` history builds | cluster |
| 2–3 | selection, SFT/QT, AGN split, **sample statistics** | anywhere (needs the HDF5s) |
| 4 | Stage 0 — per-galaxy particle files | cluster |
| 4b | annulus sampling QC — star/gas/dust counts per projected annulus × sightline | anywhere (needs Stage 0) |
| 5 | Stage 1 — selection HDF5 + Slurm masters (dust_on / dust_off) → run RT | cluster |
| 6 | aperture QC on the first `.rtout.sed` | cluster |
| 7 | Stage 2 — per-aperture flux extraction → **one catalog per aperture per dust mode** (7b″: annular CIGALE inputs) | cluster |

**Apertures & sightlines (mock observation).** Stage 0 cuts a **100 pkpc spherical region**
around each galaxy (everything: CGM, satellites, projected neighbours — a true mock aperture,
not just member particles); the RT grid spans ±100 kpc (`zoom_box_len`). Hyperion log-spaces
`N_AP = 5` projected apertures 1→100 kpc — the 10^(k/2) ladder **1, 3.16, 10, 31.6, 100 kpc**
(central → outskirts), all extracted. Each SED is peeled along **4 sightlines**
(θ,φ) = (0,0), (45,90), (90,180), (135,270) deg — one catalog per (dust mode, aperture,
inclination). `N_AP/AP_MIN_KPC/AP_MAX_KPC` and `THETA_DEG/PHI_DEG` below must match the
parameter masters that the RT jobs copy. **Requires the one-time powderday patch documented
before Stage 1 (already applied on this cluster's install).**

**Flux errors.** Hyperion's Monte-Carlo SED uncertainty, read with
`get_sed(..., uncertainties=True)` and propagated through the filter convolution
(`<filter>_err` columns; NaN if a run stored no uncertainties).

# Part 0 — Setup & configuration

In [ ]:
import os
import gc
import glob
import json
import warnings
import numpy as np
import h5py
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table, vstack, join
from astropy import units as u
from astropy.cosmology import Planck15 as COSMO   # matches the quenching machinery

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen
from simbanator.analysis.quenching import find_quenching_times

# ── simulation ────────────────────────────────────────────────────────────────
SIM_NAME = "cis25"        # SIMBA high-res 25 Mpc/h box (m25n512); must exist in ~/.simbanator/config.json
try:
    sim = Simulation(SIM_NAME)
except KeyError as e:
    raise KeyError(
        f"'{SIM_NAME}' is not registered in ~/.simbanator/config.json on this machine.\n"
        "Register it once (adjust paths to where the 25 Mpc snapshots+catalogs live):\n"
        "  from simbanator.io.config import add_simulation\n"
        "  add_simulation('cis25', data_dir='<...>/SIMBA_25/s25',\n"
        "                 catalog_dir='<...>/SIMBA_25/s25/Groups',\n"
        "                 file_format='m25n512_{snap:03d}.hdf5')\n"
        "then add \"snap_z_map\": \"zsnap_map_caesar_box100.txt\" to that entry "
        "(SIMBA boxes share the snapshot schedule)."
    ) from e
if sim.scale_factors is None:
    raise ValueError(f"'{SIM_NAME}' config has no snap_z_map — add "
                     '"snap_z_map": "zsnap_map_caesar_box100.txt" to its entry in ~/.simbanator/config.json')

# filtered-particle filename prefix (Stage 0 == Stage 1, never let them drift)
PARTICLE_PREFIX = sim.file_format.split("_{")[0]        # 'm25n512'

# ── selection: quenched + massive + realistically gas-populated ───────────────
TARGET_REDSHIFTS = [0.3, 0.6, 0.7, 1.0, 2.0]
MASS_FLOOR       = 10.0        # log10(M*/Msun) > 10
PASSIVE_FACTOR   = 0.2         # passive if sSFR < 0.2 / t_H  (== the QT threshold of find_quenching_times)
NGAS_MIN         = 21          # STRICTLY > 20 gas particles at the anchor
NSTAR_MIN        = 20          # star-particle floor (same as quench_mode_vs_sigma_gas)

# ── AGN / coupling constants (identical to quench_mode_vs_sigma_gas §0) ──────
JET_LOGMBH    = 7.5            # jet mode: log10(M_BH) > 7.5 ...
JET_FEDD      = 0.2            #           ... AND f_Edd < 0.2
XRAY_FEDD_MAX = 0.02           # (kept for reference; xcoup uses the f_gas gate)
XRAY_FGAS_MAX = 0.2            # coupling gate: f_gas = Mgas/M* < 0.2
GYR = 1e9

# ── history tracking ──────────────────────────────────────────────────────────
TRACK_AGE_FRAC      = 0.09     # track back to ~this fraction of the cosmic age at selection
ANCHOR_END_OVERRIDE = {}
CORRUPT_SNAPS       = set()

# ── heavy-build gates (set True on the cluster, then reuse the cached HDF5s) ──
BUILD_MULTI_Z = False          # per-anchor progenitor FITS + property history HDF5
BUILD_BH      = False          # per-anchor BH (mass / mdot / f_Edd) history HDF5

# ── apertures (MUST match SED_APERTURE_* in simbanator/sed/parameters_master*.py) ──
N_AP       = 5           # SED_APERTURE_NAP: 10^(k/2) ladder -> 1, 3.16, 10, 31.6, 100 kpc
AP_MIN_KPC = 1.0
AP_MAX_KPC = 100.0
APERTURE_RADII_KPC = np.geomspace(AP_MIN_KPC, AP_MAX_KPC, N_AP)
# central -> outskirts; ALL rungs are extracted (nominal labels, true radii above)
TARGET_AP_KPC   = [1, 3, 10, 32, 100]
WANTED_AP_IDX   = list(range(N_AP))
APERTURE_LABELS = [f"ap{t:g}kpc" for t in TARGET_AP_KPC]

# ── viewing angles (MUST match THETA/PHI in the parameter masters) ──
THETA_DEG   = [0, 45, 90, 135]
PHI_DEG     = [0, 90, 180, 270]
N_INCL      = len(THETA_DEG)
INCL_LABELS = [f"i{t:g}p{p:g}" for t, p in zip(THETA_DEG, PHI_DEG)]   # i0p0, i45p90, ...

# ── Stage-0 region cutout: EVERYTHING (CGM, satellites) within this proper radius ──
# sphere radius = zoom_box_len = largest aperture (100 kpc): the grid's inscribed sphere
# is fully populated; only the outermost aperture is slightly depth-truncated at its edge
R_CUTOUT_KPC = 100.0

# ── powderday run layout (same conventions as test_powderday.ipynb) ──────────
GVFS_BASE   = ''
# '+' not os.path.join: with GVFS_BASE='' this must stay ABSOLUTE (see test_powderday)
REMOTE_HOME = GVFS_BASE + "/mnt/home/glorenzon/analize_simba_cgm"

hydro_dir_base = os.path.join(os.getcwd(), 'output', sim.name, 'filtered_particles')
selection_file = 'selection_m25_quenched'                  # MakeSED appends '.h5'
sed_output_dir = os.path.join(REMOTE_HOME, 'output', sim.name, 'sed_quenched_regions')

RUNS = {
    'dust_on':  dict(run_tag='dusty_simdust', paramf='parameters_master.py'),
    'dust_off': dict(run_tag='nodust_1e-12',  paramf='parameters_master-nodust.py'),
}

# ── local output tree ─────────────────────────────────────────────────────────
OUT      = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR   = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
PLOTDIR  = os.path.join(OUT, "plots", "powderday_quenched")
CATDIR   = os.path.join(OUT, "sed_aperture_catalogs")
for _d in (SFHDIR, TABLEDIR, PLOTDIR, CATDIR):
    os.makedirs(_d, exist_ok=True)
SELECTION_FITS = os.path.join(TABLEDIR, "powderday_quenched_selection.fits")

def _ztag(z):
    return ("z%g" % z).replace(".", "p")

print(f"sim={sim.name}  data_dir={sim.data_dir}")
print(f"prefix={PARTICLE_PREFIX}  anchors z={TARGET_REDSHIFTS}")
print("aperture ladder [kpc]:", np.round(APERTURE_RADII_KPC, 2))
print("extracted rungs:", {l: f"{APERTURE_RADII_KPC[i]:.3g} kpc (idx {i})"
                           for l, i in zip(APERTURE_LABELS, WANTED_AP_IDX)})
print("sightlines:", INCL_LABELS, "  region cutout:", R_CUTOUT_KPC, "pkpc")
print("SED output:", sed_output_dir)

# Part 1 — Anchors & gated cluster builds

Each anchor (z ≈ 0.3, 0.6, 0.7, 1.0, 2.0 → nearest snapshot) gets its **own** progenitor table +
property history with that snapshot as row 0, and a BH history aligned to the same rows — exactly
the `quench_mode_vs_sigma_gas.ipynb` machinery, pointed at `cis25`. Histories are pre-selected to
**massive + passive** at the anchor (the gas/star floors are applied later so the statistics can
count them).

In [ ]:
# ── anchor table: snapshot, track end, per-anchor product paths ──
_sall, _zall = [], []
for _s in range(0, 152):
    try:
        _zv = float(sim.get_z_from_snap(_s))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)
_aall = COSMO.age(_zall).value

ANCHORS = {}
for _zt in TARGET_REDSHIFTS:
    _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
    _age_end = TRACK_AGE_FRAC * float(_aall[_sall == _snap][0])
    _end = int(ANCHOR_END_OVERRIDE.get(_zt, int(_sall[np.searchsorted(_aall, _age_end)])))
    _tag = _ztag(_zt)
    ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap,
                        z=float(sim.get_z_from_snap(_snap)), end_snap=_end,
                        prog_file=f"progenitors_anchor_{_tag}.fits",
                        hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"),
                        bh_path=os.path.join(SFHDIR, f"bh_history_anchor_{_tag}.hdf5"))

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'end':>5s} {'hist':>6s} {'BH':>4s}")
for _zt, A in ANCHORS.items():
    print(f"{_zt:6.1f} {A['snap']:5d} {A['z']:7.3f} {A['end_snap']:5d} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(A['bh_path']) else '--':>4s}")

In [ ]:
# ── property list tracked per anchor (superset of what selection + coupling need) ──
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust", "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass",
        "pos", "ngas", "nstar", "ages.mass_weighted",
    ],
    "halo_data": ["masses.total"],
}

# ── GATED (cluster): per-anchor progenitor table + property history ──
# Verbatim port of quench_mode_vs_sigma_gas 1z·build, with the (stricter) M*>10 pre-selection.
if BUILD_MULTI_Z:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['hist_path'])}"); continue
        end = int(A["end_snap"])
        while end < A["snap"] and (end in CORRUPT_SNAPS or not os.path.exists(sim.get_caesar_file(end))):
            end += 1
        A["end_snap"] = end
        print(f"[{A['tag']}] anchor snap {A['snap']} (z={A['z']:.2f}) <- {end}: progenitor table ...")
        cs_a = sim.load_catalog(snap=A["snap"])
        caesar_read_progen([g.GroupID for g in cs_a.galaxies], A["prog_file"],
                           range(end, A["snap"] + 1), sim, output_dir=None)
        hist = HDF5BuildHistory(sim, cs_a, progfilename=A["prog_file"])
        with fits.open(hist.progen_file) as hdul:
            valid_ids = np.asarray(hdul[1].data["GroupID"])
            _tHa = COSMO.age(float(A["z"])).value * 1e9
            _gid = np.array([g.GroupID for g in cs_a.galaxies])
            _ms  = np.array([float(g.masses["stellar"]) for g in cs_a.galaxies])
            _sf  = np.array([float(g.sfr) for g in cs_a.galaxies])
            with np.errstate(all="ignore"):
                _ss = np.where(_ms > 0, _sf / _ms, np.nan)
                _ok = (np.log10(np.where(_ms > 0, _ms, np.nan)) > MASS_FLOOR) & (_ss < PASSIVE_FACTOR / _tHa)
            _keep = {int(g) for g in _gid[_ok]}
            valid_ids = np.asarray([i for i in valid_ids if int(i) in _keep], dtype=valid_ids.dtype)
            print(f"  [pre-select] {len(valid_ids)}/{len(_gid)} massive+passive at z={A['z']:.2f}")
        hist.get_history_indx(valid_ids, A["snap"], end)
        props_try = {k: list(v) for k, v in PROPS.items()}
        while True:   # drop-and-retry: some catalog versions miss some fields
            try:
                hist.get_property_history(props_try, verbose=0); break
            except KeyError as e:
                msg = str(e); dropped = False
                for fam, plist in props_try.items():
                    for pr in list(plist):
                        if pr in msg or pr.split("/")[-1] in msg:
                            plist.remove(pr); print("  [drop]", pr); dropped = True
                if not dropped:
                    raise
        hist.save_history_to_hdf5(os.path.basename(A["hist_path"]))
        del cs_a, hist; gc.collect()
        print(f"[{A['tag']}] history -> {A['hist_path']}")
else:
    print("BUILD_MULTI_Z=False -> expecting per-anchor histories under", SFHDIR)

In [ ]:
# ── loaders (verbatim from quench_mode_vs_sigma_gas): row 0 = the anchor epoch ──
def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H

def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP; gc.collect()
    return M

In [ ]:
# ── BH history: per-anchor build (GATED) + loader (verbatim quench_mode §4b) ──
BH_CANDIDATES = {"bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
                 "bh_mdot": ["bhmdot", "bh_mdot"],
                 "bh_fedd": ["bh_fedd", "bhfedd", "fedd"]}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

def build_bh_for_anchor(A, galaxy_ids, snaps_arr, n_gal):
    pidx = build_prog_index(A, galaxy_ids, snaps_arr)
    n_snap = len(snaps_arr)
    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri]); cv = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    if p is not None:
                        BH[k][ri, cv] = f[p][:][vi]
        except (OSError, KeyError) as e:
            print(f"  [skip] snap {snap}: {type(e).__name__}"); CORRUPT_SNAPS.add(snap)
    with h5py.File(A["bh_path"], "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print(f"[{A['tag']}] BH history -> {A['bh_path']}")
    return BH

def load_bh(bh_hist_path):
    with h5py.File(bh_hist_path, "r") as f:
        return {k: f[k][:] for k in f.keys()}

if BUILD_BH:
    for _zt, A in ANCHORS.items():
        if os.path.exists(A["bh_path"]):
            print(f"[{A['tag']}] cached -> {os.path.basename(A['bh_path'])}"); continue
        if not os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] no history yet -> run BUILD_MULTI_Z first"); continue
        _H = load_anchor_history(A)
        build_bh_for_anchor(A, _H["galaxy_ids"], _H["snaps_arr"], len(_H["galaxy_ids"]))
        del _H; gc.collect()
else:
    print("BUILD_BH=False -> expecting per-anchor BH histories under", SFHDIR)

# Part 2 — Selection, quench events (SFT/QT) & the weak/strong AGN split

- **Selection** (at row 0 = the anchor): `log10 M* > 10`, passive (`sSFR < 0.2/t_H`), `ngas > 20`,
  `nstar ≥ 20`.
- **SFT/QT** per galaxy from `find_quenching_times` on the tracked sSFR history (SFT = crossing
  below 1/t, QT = subsequent crossing below 0.2/t with persistence); the **last** event is kept.
- **AGN split**: `xstr_quench` = mean of `xcoup_hist` (jet strength `clip(log10(0.2/f_Edd),0,1)`
  for `log M_BH > 7.5`, gated by `f_gas < 0.2`) over snapshots with `t_SFT ≤ t ≤ t_QT`; if the
  window is narrower than the snapshot spacing, the finite snapshot nearest SFT is used.
  **strong / weak = top / bottom terciles** of `xstr_quench` (per anchor); middle tercile =
  `intermediate`; no finite coupling = `no_AGN`; no detected quench event = `no_event`.

In [ ]:
# ── selection mask at the anchor epoch (row 0) ──
def selection_mask(P, t_cosmic_yr):
    mstar0 = P["masses.stellar"][0]
    sfr0   = P["sfr"][0]
    ngas0  = P["ngas"][0]
    nstar0 = P["nstar"][0] if "nstar" in P else np.full_like(mstar0, np.inf)
    with np.errstate(all="ignore"):
        ssfr0 = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)
        cuts = {
            "massive":  np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR,
            "passive":  ssfr0 < (PASSIVE_FACTOR / t_cosmic_yr[0]),
            "gas>20":   ngas0 >= NGAS_MIN,
            "star>=20": nstar0 >= NSTAR_MIN,
        }
    m = cuts["massive"] & cuts["passive"] & cuts["gas>20"] & cuts["star>=20"]
    return m, cuts

# ── SFT/QT per selected galaxy (trimmed from quench_mode build_records) ──
def quench_records(P, t_cosmic_yr, redshift, galaxy_ids, cols):
    """One record per selected column; galaxies without a detected quench event keep NaN times."""
    records = []
    for col in np.asarray(cols, int):
        gid = galaxy_ids[col]
        mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
        with np.errstate(all="ignore"):
            ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
        valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
        rec = dict(gid=int(gid), col=int(col), t_sft=np.nan, t_qt=np.nan,
                   tau_q=np.nan, tau_q_over_tH=np.nan, z_qt=np.nan)
        if valid.sum() >= 5:
            t = t_cosmic_yr[valid]; s = ssfr[valid]
            o = np.argsort(t); t, s = t[o], s[o]
            tu, ui = np.unique(t, return_index=True); su = s[ui]
            if len(tu) >= 5:
                qts, sfts, _, dbg = find_quenching_times(
                    tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True)
                if len(qts):
                    k = int(np.argmax(qts))                     # last (surviving) quench event
                    rec["t_qt"], rec["t_sft"] = float(qts[k]), float(sfts[k])
                    rec["tau_q"] = rec["t_qt"] - rec["t_sft"]
                    z_qt = float(np.interp(rec["t_qt"], t_cosmic_yr[::-1], redshift[::-1]))
                    rec["z_qt"] = z_qt
                    rec["tau_q_over_tH"] = rec["tau_q"] / (COSMO.age(z_qt).value * 1e9)
        records.append(rec)
    return records

In [ ]:
# ── AGN–ISM coupling over the quench window [SFT, QT] (physics verbatim from §8j build_coupling) ──
def coupling_quench_window(BH, P, records, t_cosmic_yr):
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    with np.errstate(all="ignore"):
        fgas_hist = np.where(P["masses.stellar"] > 0, P["masses.gas"] / P["masses.stellar"], np.nan)
        _bh_ok  = np.isfinite(BH["bh_mass"]) & np.isfinite(BH["bh_fedd"])
        _mbh_ok = BH["bh_mass"] > 10 ** JET_LOGMBH
        wjet_hist = np.where(_bh_ok, np.where(_mbh_ok,
                             np.clip(np.log10(JET_FEDD / np.clip(BH["bh_fedd"], 1e-12, None)), 0.0, 1.0),
                             0.0), np.nan)
        xcoup_hist = np.where(np.isfinite(wjet_hist) & np.isfinite(fgas_hist),
                              wjet_hist * (fgas_hist < XRAY_FGAS_MAX).astype(float), np.nan)
    n = len(records)
    xstr_q = np.full(n, np.nan)
    for i, r in enumerate(records):
        if not (np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"])):
            continue                                   # no quench event -> stays NaN ('no_event')
        cs = xcoup_hist[_ord, r["col"]].astype(float)
        fin = np.isfinite(cs)
        win = (t_inc >= r["t_sft"]) & (t_inc <= r["t_qt"]) & fin
        if not win.any() and fin.any():
            # quench window narrower than the snapshot spacing -> nearest finite snapshot to SFT
            j = np.where(fin)[0]
            win = np.zeros_like(fin); win[j[np.argmin(np.abs(t_inc[j] - r["t_sft"]))]] = True
        if win.any():
            xstr_q[i] = np.nanmean(cs[win])
    bx = np.isfinite(xstr_q)
    strong = np.zeros(n, bool); weak = np.zeros(n, bool); lo_q = hi_q = np.nan
    if bx.sum() >= 3:
        lo_q, hi_q = np.nanquantile(xstr_q[bx], [1.0 / 3.0, 2.0 / 3.0])
        strong = bx & (xstr_q >= hi_q); weak = bx & (xstr_q <= lo_q)   # §8j-style terciles
    inter = bx & ~strong & ~weak
    no_fb = ~bx
    return dict(xstr_quench=xstr_q, strong=strong, weak=weak, inter=inter, no_fb=no_fb,
                tercile=(lo_q, hi_q))

def agn_class_labels(CO, records):
    """Per-record string label; galaxies without a quench event are 'no_event'."""
    n = len(records)
    has_event = np.array([np.isfinite(r["t_sft"]) and np.isfinite(r["t_qt"]) for r in records])
    lab = np.array(["unclassified"] * n, dtype=object)
    if CO is not None:
        lab[CO["no_fb"]] = "no_AGN"
        lab[CO["inter"]] = "intermediate"
        lab[CO["weak"]]  = "weak"
        lab[CO["strong"]] = "strong"
    lab[~has_event] = "no_event"
    return lab

In [ ]:
# ── driver: per anchor -> selection, records, coupling, labels ──
RESULTS = {}
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        print(f"[{A['tag']}] MISSING history -> run BUILD_MULTI_Z on the cluster; skipped")
        continue
    H = load_anchor_history(A)
    m, cuts = selection_mask(H["P"], H["t_cosmic_yr"])
    cols = np.where(m)[0]
    recs = quench_records(H["P"], H["t_cosmic_yr"], H["redshift"], H["galaxy_ids"], cols)
    BH = load_bh(A["bh_path"]) if os.path.exists(A["bh_path"]) else None
    CO = coupling_quench_window(BH, H["P"], recs, H["t_cosmic_yr"]) if BH is not None else None
    labels = agn_class_labels(CO, recs)
    if BH is None:
        print(f"[{A['tag']}] WARNING: no BH history -> AGN split = 'unclassified' (run BUILD_BH)")
    RESULTS[_zt] = dict(A=A, H=H, mask=m, cuts=cuts, cols=cols, records=recs, CO=CO, labels=labels)
    n_ev = int(np.isfinite([r["t_qt"] for r in recs]).sum())
    print(f"[{A['tag']}] snap {A['snap']} (z={A['z']:.3f}): pool={m.size} "
          f"selected={len(cols)} with_event={n_ev} "
          f"classes={dict(zip(*np.unique(labels, return_counts=True))) if len(labels) else {}}")

# Part 3 — Sample statistics & the selection catalog

How many galaxies survive each cut per snapshot, how many have gas at all, and how the AGN classes
populate. **Note:** the pool is the history's build-time pre-selection (massive + passive at the
anchor), not the full galaxy catalog — the funnel starts there. Also writes the per-galaxy
selection table (`powderday_quenched_selection.fits`) that Stages 0–2 read, so the RT stages never
depend on this session's memory.

In [ ]:
# ── funnel table + per-galaxy selection FITS ──
_rows, _sel_rows = [], []
for _zt, R in RESULTS.items():
    A, H, cuts = R["A"], R["H"], R["cuts"]
    ngas0 = H["P"]["ngas"][0]
    n_pool = int(np.isfinite(H["P"]["masses.stellar"][0]).sum())
    lab = R["labels"]
    _rows.append(dict(
        z_target=_zt, snap=A["snap"], z_snap=round(A["z"], 4),
        pool_massive_passive=n_pool,
        with_any_gas=int((ngas0 > 0).sum()),
        gas_gt20=int(cuts["gas>20"].sum()),
        massive=int(cuts["massive"].sum()),
        passive=int(cuts["passive"].sum()),
        star_ge20=int(cuts["star>=20"].sum()),
        selected=len(R["cols"]),
        with_event=int(np.isfinite([r["t_qt"] for r in R["records"]]).sum()),
        strong=int((lab == "strong").sum()), weak=int((lab == "weak").sum()),
        intermediate=int((lab == "intermediate").sum()), no_AGN=int((lab == "no_AGN").sum()),
        no_event=int((lab == "no_event").sum()), unclassified=int((lab == "unclassified").sum()),
    ))
    # per-galaxy rows
    P0 = H["P"]
    for i, (r, l) in enumerate(zip(R["records"], lab)):
        c = r["col"]
        with np.errstate(all="ignore"):
            _ms = float(P0["masses.stellar"][0, c])
            _sf = float(P0["sfr"][0, c])
            xs = R["CO"]["xstr_quench"][i] if R["CO"] is not None else np.nan
        _sel_rows.append(dict(
            snap=int(A["snap"]), z_snap=float(A["z"]), z_target=float(_zt),
            gal_id=int(r["gid"]),
            log_mstar=float(np.log10(_ms)) if _ms > 0 else np.nan,
            ssfr=float(_sf / _ms) if _ms > 0 else np.nan,
            ngas=int(P0["ngas"][0, c]), nstar=int(P0["nstar"][0, c]) if "nstar" in P0 else -1,
            t_sft=r["t_sft"], t_qt=r["t_qt"], tau_q=r["tau_q"],
            tau_q_over_tH=r["tau_q_over_tH"], z_qt=r["z_qt"],
            xstr_quench=float(xs), agn_class=str(l),
        ))

STATS = Table(_rows)
STATS.write(os.path.join(TABLEDIR, "powderday_quenched_stats.fits"), overwrite=True)
STATS.pprint(max_width=-1)

SEL = Table(_sel_rows)
SEL.write(SELECTION_FITS, overwrite=True)
print(f"\nselection table: {len(SEL)} galaxies over {len(np.unique(SEL['snap']))} snapshots "
      f"-> {SELECTION_FITS}")

In [ ]:
# ── figures: selection funnel + gas-particle content + AGN classes ──
_zs   = list(RESULTS.keys())
_tags = [RESULTS[z]["A"]["tag"] for z in _zs]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# funnel per anchor
_steps = ["pool_massive_passive", "with_any_gas", "gas_gt20", "selected", "with_event"]
_slbl  = ["massive+passive", "any gas", "gas>20", "all cuts", "SFT/QT found"]
_x = np.arange(len(_zs)); _w = 0.16
for j, (st, sl) in enumerate(zip(_steps, _slbl)):
    axes[0].bar(_x + (j - 2) * _w, [STATS[st][i] for i in range(len(STATS))], width=_w, label=sl)
axes[0].set_xticks(_x); axes[0].set_xticklabels(_tags)
axes[0].set_ylabel("N galaxies"); axes[0].set_title("selection funnel")
axes[0].legend(fontsize=9)

# gas-particle histograms (pool), with the >20 floor
for z in _zs:
    ng = RESULTS[z]["H"]["P"]["ngas"][0]
    ng = ng[np.isfinite(ng) & (ng > 0)]
    if ng.size:
        axes[1].hist(np.log10(ng), bins=25, histtype="step", lw=2, label=RESULTS[z]["A"]["tag"])
axes[1].axvline(np.log10(NGAS_MIN), color="k", ls=":", label=f"ngas={NGAS_MIN}")
axes[1].set_xlabel("log10 ngas (anchor)"); axes[1].set_ylabel("N")
axes[1].set_title("gas-particle content of the pool"); axes[1].legend(fontsize=9)

# AGN classes among the selected
_classes = ["strong", "intermediate", "weak", "no_AGN", "no_event", "unclassified"]
_bot = np.zeros(len(_zs))
for cl in _classes:
    v = np.array([STATS[cl][i] for i in range(len(STATS))], float)
    axes[2].bar(_x, v, bottom=_bot, label=cl)
    _bot += v
axes[2].set_xticks(_x); axes[2].set_xticklabels(_tags)
axes[2].set_ylabel("N selected"); axes[2].set_title("AGN-coupling classes (quench window)")
axes[2].legend(fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(PLOTDIR, "sample_statistics.png"), dpi=150, bbox_inches="tight")
plt.show()

# Part 3b — mass–size QC: flag sources too large (or too small) for the apertures

Fixed **physical** apertures implicitly assume every source has a similar size — the mass–size
relation is the check on that. Anchor-epoch CAESAR radii come from the histories (row 0;
`radii.*` are **comoving kpc** — verified `unit: 'kpccm'` in the m25n512 catalogs — converted
with $1/(1+z)$). CAESAR $R_{50}$ is the 3D half-**mass** radius; the van der Wel+2014 quiescent
relations (projected half-light $R_e$) are drawn for context only.

Flags (written back into `SELECTION_FITS`; Part 7 carries them into every catalog):

- **`flag_too_large`** — `SIZE_FACTOR·R50 > R_CUTOUT_KPC`: the 100 pkpc cutout/grid truncates
  the stellar envelope → the "≈ total" 100 kpc aperture (and any CIGALE mass) biases low;
- **`flag_unresolved`** — `R50 < N_EPS_MIN·ε` (softening): the size is not trusted and the
  1 kpc "central" aperture is not meaningfully sub-galactic.

Nothing is dropped — the flags are one boolean away in any downstream cut. The per-rung print
shows for how much of the sample each aperture is sub-galactic (< R50) vs effectively total
(> 3 R50).


In [ ]:
# ── Part 3b — mass-size QC: flag too-large / unresolved sources for the aperture ladder ──
# Self-contained after Part 0: reads SELECTION_FITS + the anchor histories in SFHDIR.
SIZE_FACTOR    = 5.0     # envelope proxy: SIZE_FACTOR*R50 beyond the cutout -> truncated
N_EPS_MIN      = 2.0     # resolved if R50 >= N_EPS_MIN * softening
EPS_MIN_CKPC_H = 0.25    # m25n512 minimum gravitational softening [comoving kpc/h]
SIMBA_H        = 0.68

SEL = Table.read(SELECTION_FITS)
_rdb = {}
for _hf in sorted(glob.glob(os.path.join(SFHDIR, "history_anchor_*.hdf5"))):
    with h5py.File(_hf, "r") as f:
        _snap0 = int(f["metadata/snapshots"][0])               # row 0 = anchor epoch
        _gid   = np.asarray(f["metadata/galaxy_ids"][:], int)
        _z0    = float(f["redshift/Redshift"][0])
        _r0    = {k: f[f"properties/{k}"][0] for k in
                  ("radii.stellar_half_mass", "radii.gas_half_mass")
                  if f"properties/{k}" in f}
    for _j, _g in enumerate(_gid):                             # kpccm -> proper kpc
        _rdb[(_snap0, int(_g))] = {k: float(v[_j]) / (1.0 + _z0) for k, v in _r0.items()}

_r50s = np.full(len(SEL), np.nan); _r50g = np.full(len(SEL), np.nan)
for _k, (_s, _g) in enumerate(zip(SEL["snap"], SEL["gal_id"])):
    _r = _rdb.get((int(_s), int(_g)), {})
    _r50s[_k] = _r.get("radii.stellar_half_mass", np.nan)
    _r50g[_k] = _r.get("radii.gas_half_mass", np.nan)

_zsel = np.asarray(SEL["z_snap"], float)
_eps_pkpc = EPS_MIN_CKPC_H / SIMBA_H / (1.0 + _zsel)           # softening, proper kpc
flag_too_large  = SIZE_FACTOR * _r50s > R_CUTOUT_KPC
flag_unresolved = _r50s < N_EPS_MIN * _eps_pkpc

SEL["r50_star_kpc"]    = _r50s
SEL["r50_gas_kpc"]     = _r50g
SEL["flag_too_large"]  = flag_too_large.astype(int)
SEL["flag_unresolved"] = flag_unresolved.astype(int)
SEL.write(SELECTION_FITS, overwrite=True)

_nok = int(np.isfinite(_r50s).sum())
print(f"R50 matched: {_nok}/{len(SEL)} | median R50 = {np.nanmedian(_r50s):.2f} pkpc | "
      f"too large (R50 > {R_CUTOUT_KPC/SIZE_FACTOR:.0f} kpc): {int(flag_too_large.sum())} | "
      f"unresolved (R50 < {N_EPS_MIN:g} eps): {int(flag_unresolved.sum())}")
print("aperture rung vs the sample sizes:")
_rungs = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]
for _t, _rr in zip(APERTURE_LABELS, _rungs):
    _sub = 100 * np.nanmean(_rr < _r50s); _tot = 100 * np.nanmean(_rr > 3 * _r50s)
    print(f"  {_t:>9s} ({_rr:6.2f} kpc): sub-galactic (<R50) for {_sub:4.0f}%  |  "
          f"~total (>3 R50) for {_tot:4.0f}%")
for _k in np.where(flag_too_large | flag_unresolved)[0]:
    _why = "TOO LARGE" if flag_too_large[_k] else "unresolved"
    print(f"  [{_why}] snap {int(SEL['snap'][_k])} gal {int(SEL['gal_id'][_k])}: "
          f"R50={_r50s[_k]:.2f} pkpc, logM*={float(SEL['log_mstar'][_k]):.2f}, "
          f"z={_zsel[_k]:.2f}")

# ── mass-size relation vs the aperture ladder ──
_fig, _ax = plt.subplots(figsize=(7.4, 5.6))
_zt_colors = plt.cm.viridis(np.linspace(0, 0.9, len(TARGET_REDSHIFTS)))
# van der Wel+2014 Table 5, early types: R_e = A*(M*/5e10)^alpha [kpc] (context only)
_VDW = {0.25: (10**0.60, 0.75), 0.75: (10**0.42, 0.71), 1.25: (10**0.22, 0.76),
        1.75: (10**0.09, 0.76), 2.25: (10**-0.05, 0.79)}
_lm = np.asarray(SEL["log_mstar"], float)
_xmax = max(11.4, np.nanmax(_lm) + 0.15)
_mm = np.logspace(10, _xmax, 40)
for _c, _zt in zip(_zt_colors, TARGET_REDSHIFTS):
    _m = np.isclose(np.asarray(SEL["z_target"], float), _zt)
    if not _m.any():
        continue
    _ax.scatter(_lm[_m], _r50s[_m], s=22, color=_c, label=f"z\u2248{_zt:g}", zorder=3)
    _A, _al = _VDW[min(_VDW, key=lambda z: abs(z - _zt))]
    _ax.plot(np.log10(_mm), _A * (_mm / 5e10) ** _al, "--", color=_c, lw=1.1, alpha=0.7)
for _k in np.where(flag_too_large | flag_unresolved)[0]:
    _ax.scatter([_lm[_k]], [_r50s[_k]], s=95, facecolor="none",
                edgecolor="crimson", lw=1.4, zorder=4)
for _rr, _t in zip(_rungs, APERTURE_LABELS):
    _ax.axhline(_rr, color="0.78", lw=0.7, zorder=1)
    _ax.text(_xmax - 0.03, _rr * 1.04, _t, fontsize=7, va="bottom", ha="right", color="0.45")
_ax.axhline(R_CUTOUT_KPC / SIZE_FACTOR, color="crimson", ls=":", lw=1.3)
_ax.text(10.02, R_CUTOUT_KPC / SIZE_FACTOR * 1.05,
         f"too large ({SIZE_FACTOR:g}\u00b7R50 > {R_CUTOUT_KPC:.0f} kpc cutout)",
         fontsize=8, color="crimson", va="bottom")
_ax.set_yscale("log"); _ax.set_xlim(9.98, _xmax)
_ax.set_xlabel(r"$\log_{10}\,M_*/M_\odot$")
_ax.set_ylabel(r"stellar $R_{50}$ [proper kpc]")
_ax.set_title("mass\u2013size QC: CAESAR 3D half-mass radii vs the aperture ladder\n"
              "(dashed: van der Wel+2014 quiescent $R_e$, projected half-light \u2014 context only)",
              fontsize=9)
_ax.legend(fontsize=8, frameon=False, loc="lower right")
plt.savefig(os.path.join(PLOTDIR, "mass_size_aperture_qc.png"), dpi=140, bbox_inches="tight")
plt.show()


# Part 4 — Stage 0: extract the per-galaxy particle files (cluster)

One HDF5 per galaxy (gas + stars; gas keeps `Dust_Masses`) under
`hydro_dir_base/snap_NNN/<PREFIX>_snap<NNN>_gal<ID>.h5` — now a **100 pkpc spherical region
cutout** around each galaxy centre (CGM + satellites included; periodic-wrap safe), NOT just
the caesar member particles. Identical for `dust_on` and `dust_off` (the dust treatment lives
in the parameter master). Reads `SELECTION_FITS`, so it can run in a fresh session once Part 3
has been executed.

⚠ Region files reuse the plist filenames, so `EXTRACT_OVERWRITE = True` below **replaces** any
old galaxy-member-only files — intended, since mixed hydro inputs would corrupt the sample.

In [ ]:
from simbanator.analysis import extract_particles

SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)
print(f"{len(SNAPS)} sources over snapshots {sorted(set(SNAPS.tolist()))}")

EXTRACT_OVERWRITE = True    # region cutouts REPLACE the old plist files (same names)
EXTRACT_PTYPES    = ("PartType0", "PartType4")   # gas (carries Dust_Masses) + stars

bad_snaps = []
for _snap in np.unique(SNAPS):
    _snap = int(_snap)
    _ids_here = np.unique(IDS[SNAPS == _snap])
    _simfile = sim.get_snapshot_file(_snap)
    print(f"snap {_snap:3d}: extracting {len(_ids_here)} galaxies from {os.path.basename(_simfile)}")
    try:
        _cs = sim.load_catalog(snap=_snap)
        extract_particles(_cs, _simfile, _snap, galaxy_ids=_ids_here, radius=R_CUTOUT_KPC,
                          ptypes=EXTRACT_PTYPES, sim_name=sim.name, prefix=PARTICLE_PREFIX,
                          overwrite=EXTRACT_OVERWRITE, verbose=1)
        del _cs
    except (OSError, KeyError) as e:
        print(f"  [SKIP] snap {_snap}: {type(e).__name__}: {str(e).splitlines()[0]}")
        bad_snaps.append((_snap, len(_ids_here)))

print("\nparticle extraction complete ->", hydro_dir_base)
if bad_snaps:
    print(f"{len(bad_snaps)} snapshot(s) unreadable: {bad_snaps} — re-stage those files and re-run.")

# Part 4b — annulus sampling QC: particle counts per projected annulus

How well can powderday sample an **annular** SED? Each Hyperion aperture is a circle in the
**image plane**, so for every sightline the star (emission sources) and gas (dust carriers)
particles of each 100 pkpc cutout are projected perpendicular to the viewing direction and
counted in the annuli between consecutive rungs (`ann1kpc` = the 0→1 kpc disc, then 1→3.16,
3.16→10, 10→31.6, 31.6→100 kpc; as in Part 7a′, the **outer** rung names the annulus).
Counts span the whole LOS depth through the sphere — exactly the geometry the RT sees (the
outermost annulus is depth-truncated like its aperture). Centres are the **exact RT grid
centres** (`code_coods` from the Stage-1 selection HDF5; caesar fallback if Part 5 has not
run yet).

Reading the numbers:

- **stars = intrinsic emitters.** `nstar = 0` → the annular flux is scattered/re-emitted
  light only and a CIGALE fit of it is meaningless; a few tens of star particles → the
  annular SED is shot-noise dominated (a handful of SSP ages/metallicities).
- **gas → dust grid.** Gas is smoothed onto the octree, so counts are indicative; `ndust`
  (gas with `Dust_Masses > 0`) counts the particles actually carrying dust.

One row per (galaxy, sightline) → `tables/annulus_particle_counts.fits`
(`nstar_/ngas_/ndust_/ntot_<annulus>` + `A_V_glob`/`dusty`); Part 7b″ reads it to flag
star-free annular catalogs.

**Dusty vs non-dusty split.** If Part 7a′'s `attenuation_vs_ism.fits` exists, each galaxy is
flagged **dusty** (global $A_V > 0.1$, same threshold as 7a′ Fig 3, fiducial aperture +
sightline) or non-dusty, the figure highlights the two subsamples (red vs gray lines, separate
medians) and the summary prints their per-annulus median counts — the dusty galaxies are the
ones whose annular attenuation/CIGALE fits matter, so their sampling is the QC that counts.
Part 7a′ needs the RT fluxes, so on a fresh pipeline this cell first runs without the split
(`dusty = -1`) — **re-run it after Part 7a′** to get the highlighted version.

In [ ]:
# ── Part 4b — annulus sampling QC: star/gas counts per projected annulus & sightline ──
# Self-contained after Part 0 + the Stage-0 cutouts (Part 4).
_EDGES = np.concatenate([[0.0], APERTURE_RADII_KPC])            # 0, 1, 3.16, 10, 31.6, 100 pkpc
_R_MID = np.where(_EDGES[:-1] > 0, np.sqrt(_EDGES[:-1] * _EDGES[1:]), _EDGES[1:] / 2.0)
ANN_LABELS = [l.replace("ap", "ann") for l in APERTURE_LABELS]  # outer rung names the annulus
_th, _ph = np.deg2rad(THETA_DEG), np.deg2rad(PHI_DEG)
_NHAT = np.column_stack([np.sin(_th) * np.cos(_ph),
                         np.sin(_th) * np.sin(_ph), np.cos(_th)])   # LOS unit vectors

SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)

# exact RT grid centres (code units): Stage-1 selection h5, else caesar (identical values)
_centers = {}
_selh5 = os.path.join(sed_output_dir, RUNS['dust_on']['run_tag'],
                      'target_selection', selection_file + '.h5')
if os.path.exists(_selh5):
    with h5py.File(_selh5, "r") as f:
        for _grp in f:
            for _g, _p in zip(f[_grp]['galaxy_GroupID'][:], f[_grp]['code_coods'][:]):
                _centers[(int(_grp[4:]), int(_g))] = np.asarray(_p, float)
    print(f"grid centres from the Stage-1 selection h5 ({len(_centers)} galaxies)")
else:
    print(f"[fallback] {_selh5} missing -> reading centres from the caesar catalogs")
    for _s in np.unique(SNAPS):
        _cs = sim.load_catalog(snap=int(_s))
        for _g in np.unique(IDS[SNAPS == _s]):
            _centers[(int(_s), int(_g))] = _cs.galaxies[int(_g)].pos.in_units('code_length').value
        del _cs

_rows, _skipped = [], []
for _s, _g in zip(SNAPS, IDS):
    _f = os.path.join(hydro_dir_base, f"snap_{_s:03d}",
                      f"{PARTICLE_PREFIX}_snap{_s:03d}_gal{_g:06d}.h5")
    _c = _centers.get((int(_s), int(_g)))
    if _c is None or not os.path.exists(_f):
        _skipped.append((int(_s), int(_g)))
        continue
    with h5py.File(_f, "r") as f:
        _a  = float(f["Header"].attrs["Time"])                    # scale factor
        _hh = float(f["Header"].attrs["HubbleParam"])
        _d = {"star": (f["PartType4/Coordinates"][:] - _c) * _a / _hh}   # ckpc/h -> proper kpc
        _gpos = (f["PartType0/Coordinates"][:] - _c) * _a / _hh
        _dm = (f["PartType0/Dust_Masses"][:] if "Dust_Masses" in f["PartType0"]
               else np.zeros(len(_gpos)))
    _d["gas"]  = _gpos
    _d["dust"] = _gpos[np.asarray(_dm) > 0]                       # dust-carrying gas
    for _j, _il in enumerate(INCL_LABELS):
        _v = _NHAT[_j]
        _row = {"snap": int(_s), "gal_id": int(_g), "incl": _il}
        for _pt, _pos in _d.items():                              # projected radius wrt LOS
            _R = np.sqrt(np.clip(np.einsum('ij,ij->i', _pos, _pos) - (_pos @ _v) ** 2, 0, None))
            _cnt, _ = np.histogram(_R, _EDGES)
            for _k, _al in enumerate(ANN_LABELS):
                _row[f"n{_pt}_{_al}"] = int(_cnt[_k])
        for _al in ANN_LABELS:
            _row[f"ntot_{_al}"] = _row[f"nstar_{_al}"] + _row[f"ngas_{_al}"]
        _rows.append(_row)

QC_COUNTS = Table(_rows)
QC_COUNTS.meta["R_EDGES"] = list(np.round(_EDGES, 3))             # proper kpc

# dusty split from Part 7a' (global A_V, fiducial aperture/sightline):
# dusty = 1 (A_V > AV_DUSTY), 0 (transparent), -1 (no A_V yet -> re-run after Part 7a')
AV_DUSTY = 0.1                                    # same threshold as Part 7a' Fig 3
_avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
_av_db = {}
if os.path.exists(_avf):
    _at = Table.read(_avf)
    _av_db = {(int(s), int(g)): float(a) for s, g, a in
              zip(_at["snap"], _at["gal_id"], _at["A_V"])}
_avg = np.array([_av_db.get((int(s), int(g)), np.nan)
                 for s, g in zip(QC_COUNTS["snap"], QC_COUNTS["gal_id"])])
QC_COUNTS["A_V_glob"] = _avg
QC_COUNTS["dusty"] = np.where(np.isnan(_avg), -1, (_avg > AV_DUSTY).astype(int))

_out = os.path.join(TABLEDIR, "annulus_particle_counts.fits")
QC_COUNTS.write(_out, overwrite=True)
print(f"{len(QC_COUNTS)} rows ({len(QC_COUNTS)//N_INCL} galaxies x {N_INCL} sightlines) -> {_out}")
if _skipped:
    print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")
_dg = np.asarray(QC_COUNTS["dusty"], int)[::N_INCL]   # per galaxy (same on all sightlines)
if _av_db:
    print(f"dusty split (Part 7a' global A_V > {AV_DUSTY:g}): {int((_dg == 1).sum())} dusty / "
          f"{int((_dg == 0).sum())} non-dusty / {int((_dg == -1).sum())} unmatched galaxies")
else:
    print(f"[dusty split] {os.path.basename(_avf)} not found — run Part 7a' then re-run this "
          "cell to highlight dusty vs non-dusty")

# ── summary: how well is each annulus sampled? ──
print(f"\n{'annulus':>10s} {'r [pkpc]':>13s} | {'nstar p16/50/84':>17s} {'=0':>4s} {'<10':>4s} "
      f"{'<100':>5s} | {'ngas p50':>8s} {'=0':>4s} | {'ndust p50':>9s} {'=0':>4s}")
for _k, _al in enumerate(ANN_LABELS):
    _ns = np.asarray(QC_COUNTS[f"nstar_{_al}"], int)
    _ng = np.asarray(QC_COUNTS[f"ngas_{_al}"],  int)
    _nd = np.asarray(QC_COUNTS[f"ndust_{_al}"], int)
    _p  = np.percentile(_ns, [16, 50, 84]).astype(int)
    print(f"{_al:>10s} {_EDGES[_k]:5.1f}-{_EDGES[_k+1]:6.1f} | "
          f"{_p[0]:5d}/{_p[1]:5d}/{_p[2]:5d} {np.mean(_ns == 0)*100:3.0f}% {np.mean(_ns < 10)*100:3.0f}% "
          f"{np.mean(_ns < 100)*100:4.0f}% | {int(np.median(_ng)):8d} {np.mean(_ng == 0)*100:3.0f}% | "
          f"{int(np.median(_nd)):9d} {np.mean(_nd == 0)*100:3.0f}%")
_nfree = int(sum((np.asarray(QC_COUNTS[f"nstar_{_al}"], int) == 0).sum() for _al in ANN_LABELS))
print(f"\nstar-free (annulus, galaxy, sightline) triples: {_nfree} "
      f"/ {len(QC_COUNTS) * len(ANN_LABELS)} — those annular SEDs have NO intrinsic emitters")

if _av_db:                       # median counts split by the Part 7a' dusty flag
    _dm_all = np.asarray(QC_COUNTS["dusty"], int)
    print(f"\nmedian counts, dusty (D, n={int((_dg == 1).sum())} gals) "
          f"vs non-dusty (N, n={int((_dg == 0).sum())}):")
    print(f"{'annulus':>10s} | {'nstar D':>8s} {'nstar N':>8s} | {'ngas D':>8s} {'ngas N':>8s} "
          f"| {'ndust D':>8s} {'ndust N':>8s}")
    for _al in ANN_LABELS:
        _vals = []
        for _cc in ("nstar", "ngas", "ndust"):
            _v = np.asarray(QC_COUNTS[f"{_cc}_{_al}"], int)
            for _dd in (1, 0):
                _m = _dm_all == _dd
                _vals.append(int(np.median(_v[_m])) if _m.any() else -1)
        print(f"{_al:>10s} | {_vals[0]:8d} {_vals[1]:8d} | {_vals[2]:8d} {_vals[3]:8d} "
              f"| {_vals[4]:8d} {_vals[5]:8d}")

# ── figure: count distributions per annulus, dusty vs non-dusty highlighted ──
_dm_all = np.asarray(QC_COUNTS["dusty"], int)
_have_split = bool(_av_db) and (_dm_all >= 0).any()
_fig, _axs = plt.subplots(1, 3, figsize=(13.5, 4.4), sharey=True)
for _ax, _pt, _ttl in zip(_axs, ("star", "gas", "dust"),
                          ("star particles (emitters)", "gas particles",
                           "dust-carrying gas (Dust_Masses > 0)")):
    _M = np.column_stack([np.asarray(QC_COUNTS[f"n{_pt}_{_al}"], int) for _al in ANN_LABELS])
    if _have_split:                # per-(gal,sightline) lines colored by the Part 7a' split
        for _rowv, _dd in zip(_M, _dm_all):
            _ax.plot(_R_MID, _rowv, color={1: "#c0392b", 0: "0.75"}.get(_dd, "0.88"),
                     lw=0.5, alpha=0.45, zorder=1)
        for _dd, _col, _mk, _lab in ((1, "#c0392b", "o-", f"dusty ($A_V>{AV_DUSTY:g}$)"),
                                     (0, "#2980b9", "s--", "non-dusty")):
            _mrows = _dm_all == _dd
            if _mrows.any():
                _ax.plot(_R_MID, np.median(_M[_mrows], axis=0), _mk, color=_col, lw=2,
                         zorder=3, label=f"{_lab} median (n={int(_mrows.sum()) // N_INCL} gals)")
    else:
        for _rowv in _M:                                          # one line per (gal, sightline)
            _ax.plot(_R_MID, _rowv, color="0.75", lw=0.5, alpha=0.5, zorder=1)
        _ax.fill_between(_R_MID, np.percentile(_M, 16, axis=0), np.percentile(_M, 84, axis=0),
                         color="#2980b9", alpha=0.25, zorder=2, label="16–84%")
        _ax.plot(_R_MID, np.median(_M, axis=0), "o-", color="#2980b9", lw=2, zorder=3,
                 label="median")
    for _thr, _ls in ((10, ":"), (100, "--")):
        _ax.axhline(_thr, color="0.3", ls=_ls, lw=0.9)
        _ax.text(_R_MID[0] * 0.9, _thr * 1.15, f"N={_thr}", color="0.3", fontsize=7)
    _ax.set_xscale("log"); _ax.set_yscale("symlog", linthresh=1)
    _ax.set_xticks(_R_MID); _ax.set_xticklabels([l[3:] for l in ANN_LABELS], fontsize=8)
    _ax.set_xlabel("annulus (outer-rung label)"); _ax.set_title(_ttl, fontsize=10)
    _ax.set_ylim(bottom=-0.5)
_axs[0].set_ylabel("particles per projected annulus (full LOS depth)")
_axs[0].legend(fontsize=8, frameon=False, loc="upper left")
_fig.suptitle("annulus sampling QC — all galaxies x 4 sightlines"
              + (" — dusty split: Part 7a′ global $A_V$" if _have_split else ""), fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(PLOTDIR, "annulus_particle_counts.png"), dpi=140, bbox_inches="tight")
plt.show()

# Part 4c — SIMBA metallicities per aperture & annulus (CIGALE priors)

Mass-weighted **stellar** and **gas** metallicities (total metal mass fraction,
`Metallicity[:, 0]`) of each cutout, measured in the **same projected geometry as the SEDs**:
per sightline, cumulative within each aperture rung (`ap1kpc…ap100kpc`) and in each annulus
between rungs (`ann3kpc…ann100kpc`; `ann1kpc`≡`ap1kpc`). Same centres/projection as Part 4b.

These are the **metallicity priors for the CIGALE runs** (Part 7c′): CIGALE's bc03
`metallicity` and nebular `zgas` are strict grids, so each galaxy's SIMBA value is snapped to
the **nearest allowed grid value in log space** (bc03: 0.0001, 0.0004, 0.004, 0.008, 0.02,
0.05 — verified against the cluster's CIGALE 2025.1 sources) and the catalog is split into
per-metallicity sub-runs: one per bc03 node, each fitted with that single stellar Z and a
`zgas` grid restricted to the group members' snapped values. The summary below shows how
the sample maps onto the bc03 nodes per aperture — i.e. how many sub-runs Part 7c′ will
create. Empty apertures/annuli (no particles) → NaN → the `Zsfree` group (default Z grid).

One row per (galaxy, sightline) → `tables/aperture_metallicities.fits`
(`Zstar_<label>`, `Zgas_<label>` for the 9 labels).

In [ ]:
# ── Part 4c — mass-weighted Z_star / Z_gas per projected aperture & annulus ──
# Self-contained after Part 0 + the Stage-0 cutouts; same geometry as Part 4b.
from simbanator.sed.cigale import grid_options, nearest_option

_EDGES = np.concatenate([[0.0], APERTURE_RADII_KPC])
AP_LABELS_Z  = list(APERTURE_LABELS)                             # cumulative rungs
ANN_LABELS_Z = [l.replace("ap", "ann") for l in APERTURE_LABELS[1:]]   # true annuli only
Z_LABELS = AP_LABELS_Z + ANN_LABELS_Z
_th, _ph = np.deg2rad(THETA_DEG), np.deg2rad(PHI_DEG)
_NHAT = np.column_stack([np.sin(_th) * np.cos(_ph),
                         np.sin(_th) * np.sin(_ph), np.cos(_th)])

SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)

# exact RT grid centres (code units): Stage-1 selection h5, else caesar (identical values)
_centers = {}
_selh5 = os.path.join(sed_output_dir, RUNS['dust_on']['run_tag'],
                      'target_selection', selection_file + '.h5')
if os.path.exists(_selh5):
    with h5py.File(_selh5, "r") as f:
        for _grp in f:
            for _g, _p in zip(f[_grp]['galaxy_GroupID'][:], f[_grp]['code_coods'][:]):
                _centers[(int(_grp[4:]), int(_g))] = np.asarray(_p, float)
else:
    for _s in np.unique(SNAPS):
        _cs = sim.load_catalog(snap=int(_s))
        for _g in np.unique(IDS[SNAPS == _s]):
            _centers[(int(_s), int(_g))] = _cs.galaxies[int(_g)].pos.in_units('code_length').value
        del _cs

def _mwz(z, m, sel):
    # mass-weighted metallicity over a particle selection (NaN if empty)
    if not sel.any():
        return np.nan
    mm = m[sel]
    return float(np.sum(mm * z[sel]) / np.sum(mm)) if mm.sum() > 0 else np.nan

_rows, _skipped = [], []
for _s, _g in zip(SNAPS, IDS):
    _f = os.path.join(hydro_dir_base, f"snap_{_s:03d}",
                      f"{PARTICLE_PREFIX}_snap{_s:03d}_gal{_g:06d}.h5")
    _c = _centers.get((int(_s), int(_g)))
    if _c is None or not os.path.exists(_f):
        _skipped.append((int(_s), int(_g)))
        continue
    with h5py.File(_f, "r") as f:
        _a  = float(f["Header"].attrs["Time"])
        _hh = float(f["Header"].attrs["HubbleParam"])
        _P = {}
        for _pt, _nm in (("PartType4", "star"), ("PartType0", "gas")):
            _pos = (f[f"{_pt}/Coordinates"][:] - _c) * _a / _hh      # -> proper kpc
            _met = f[f"{_pt}/Metallicity"][:]
            _z = np.asarray(_met[:, 0] if _met.ndim == 2 else _met, float)  # col 0 = total Z
            _P[_nm] = (_pos, np.asarray(f[f"{_pt}/Masses"][:], float), _z)
    for _j, _il in enumerate(INCL_LABELS):
        _v = _NHAT[_j]
        _row = {"snap": int(_s), "gal_id": int(_g), "incl": _il}
        for _nm, (_pos, _m, _z) in _P.items():
            _R = np.sqrt(np.clip(np.einsum('ij,ij->i', _pos, _pos) - (_pos @ _v) ** 2,
                                 0, None))
            for _k, _lab in enumerate(AP_LABELS_Z):                  # cumulative
                _row[f"Z{_nm}_{_lab}"] = _mwz(_z, _m, _R <= _EDGES[_k + 1])
            for _k, _lab in enumerate(ANN_LABELS_Z, start=1):        # annular
                _row[f"Z{_nm}_{_lab}"] = _mwz(_z, _m,
                                              (_R > _EDGES[_k]) & (_R <= _EDGES[_k + 1]))
        _rows.append(_row)

ZTAB = Table(_rows)
ZTAB.meta["R_EDGES"] = list(np.round(_EDGES, 3))
_out = os.path.join(TABLEDIR, "aperture_metallicities.fits")
ZTAB.write(_out, overwrite=True)
print(f"{len(ZTAB)} rows ({len(ZTAB)//N_INCL} galaxies x {N_INCL} sightlines) -> {_out}")
if _skipped:
    print(f"[WARN] {len(_skipped)} galaxies without cutout/centre, skipped: {_skipped}")

# ── summary: sample vs the CIGALE grids (how many Z sub-runs Part 7c' will make) ──
ZSUN = 0.0134
_zs_grid = grid_options("bc03", "metallicity")
print(f"\nbc03 metallicity nodes: {_zs_grid}")
print(f"{'label':>10s} | {'med Z*/Zsun':>11s} {'med Zgas/Zsun':>13s} | bc03 node counts (stars)")
for _lab in Z_LABELS:
    _zsv = np.asarray(ZTAB[f"Zstar_{_lab}"], float)
    _zgv = np.asarray(ZTAB[f"Zgas_{_lab}"], float)
    _near = nearest_option(_zsv, _zs_grid)
    _cnt = {f"{_n:g}": int(np.sum(_near == _n)) for _n in _zs_grid
            if np.sum(_near == _n)}
    _nnan = int(np.sum(~np.isfinite(_near)))
    if _nnan:
        _cnt["free"] = _nnan
    print(f"{_lab:>10s} | {np.nanmedian(_zsv)/ZSUN:11.2f} {np.nanmedian(_zgv)/ZSUN:13.2f} | {_cnt}")

# Part 5 — Stage 1: selection HDF5 + Slurm masters (both dust runs)

## ⚠ REQUIRED once per powderday install: the multi-aperture patch

Stock powderday gives the peeled SED a **single infinite aperture**; the parameter masters in this
repo now carry `SED_APERTURE_NAP / SED_APERTURE_MIN_KPC / SED_APERTURE_MAX_KPC`, but powderday must
be taught to read them (**already applied** in this cluster's `powderday/front_end_tools.py`,
both SED branches — redo only on a fresh install). On the cluster, locate the peeled-image setup:

```bash
grep -rn "add_peeled_images" $(python -c "import powderday, os; print(os.path.dirname(powderday.__file__))")
```

and immediately after the image-configuration lines (`set_viewing_angles` / `set_track_origin` /
`set_uncertainties`), insert (adapt `image` / `cfg.par` to the local variable names in that file):

```python
# --- multi-aperture SEDs (analize_simba_cgm patch) ---
try:
    from hyperion.util.constants import kpc as _kpc
    _nap = int(getattr(cfg.par, 'SED_APERTURE_NAP', 0))
    if _nap > 0:
        image.set_aperture_range(_nap,
                                 float(cfg.par.SED_APERTURE_MIN_KPC) * _kpc,
                                 float(cfg.par.SED_APERTURE_MAX_KPC) * _kpc)
        image.set_uncertainties(True)   # Monte-Carlo SED errors -> <filter>_err columns
except Exception as _e:
    print('[aperture patch] skipped:', _e)
```

Hyperion **log-spaces** the apertures between min and max: 1→100 kpc with `NAP=5` gives the
10^(k/2) ladder **1, 3.16, 10, 31.6, 100 kpc** (central → outskirts).
Viewing angles need **no extra patch**: stock powderday already reads
`MANUAL_ORIENTATION / THETA / PHI` from the parameter master. Verify with the Part 6 QC cell
after the first galaxy finishes.

Then run this cell, launch `submit_all_snaps.sh` under each run's `powderday_sed_out/`, and come
back to Part 6/7 when the `.rtout.sed` files exist.

In [ ]:
# ── MakeSED handles (constructor only — cheap; Parts 6–7 need just this cell, not the next) ──
from simbanator.sed.makesed import MakeSED

makeseds = {
    key: MakeSED(sim, nnodes=1, model_run_name=cfg['run_tag'],
                 hydro_dir_base=hydro_dir_base, selection_file=selection_file,
                 output_dir=sed_output_dir, run_tag=cfg['run_tag'])
    for key, cfg in RUNS.items()
}
for key, ms in makeseds.items():
    print(f"{key:9s} -> run_tag='{ms.run_tag}', master='{RUNS[key]['paramf']}'")

In [ ]:
# ── write the selection HDF5 + generate the Slurm masters (run once per sample change) ──
SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)

for key, cfg in RUNS.items():
    ms = makeseds[key]
    print(f"\n=== {key} (run_tag='{ms.run_tag}', master='{cfg['paramf']}') ===")
    ms.selection_gals(snaps=SNAPS, galaxyID=IDS)                 # same sources for both runs
    ms.create_master('cluster', 'region', radius=R_CUTOUT_KPC,
                     partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
                     prefix=PARTICLE_PREFIX, paramf=cfg['paramf'], snaps_to_run=None)

# Part 6 — Aperture QC (run after the first `.rtout.sed` exists)

Confirms the powderday patch took effect **before** burning time on the full extraction:
reads the aperture layout stored in one output file, probes every aperture index, and prints
the expected index → radius mapping.

In [ ]:
from simbanator.sed.makesed import list_sed_apertures, _read_sed

_pat = os.path.join(makeseds['dust_on'].model_dir_base, 'snap_*', 'gal_*', '*.rtout.sed')
_cands = sorted(glob.glob(_pat))
if not _cands:
    raise FileNotFoundError(f"no .rtout.sed yet under {makeseds['dust_on'].model_dir_base} — "
                            "run the RT jobs first")
_probe = _cands[0]
print("probing:", _probe, "\n")

for gname, entry in list_sed_apertures(_probe).items():
    print(f"[{gname}] seds shape = {entry.get('seds_shape')}")
    for k, v in entry.get('seds_attrs', {}).items():
        print(f"    seds.attrs[{k!r}] = {v}")
    for k, v in entry['group_attrs'].items():
        print(f"    group.attrs[{k!r}] = {v}")

print("\nexpected mapping (log-spaced, from the parameter master):")
for i, r in enumerate(APERTURE_RADII_KPC):
    mark = (f"   <- {APERTURE_LABELS[WANTED_AP_IDX.index(i)]}"
            if i in WANTED_AP_IDX else "")
    print(f"  aperture={i:2d} -> {r:7.2f} kpc{mark}")

n_ok = 0
for i in range(N_AP):
    try:
        wav, flx, unc = _read_sed(_probe, aperture=i, uncertainties=True)
        assert np.shape(flx)[0] == N_INCL, (
            f"{np.shape(flx)[0]} inclination(s) in the rtout but N_INCL={N_INCL} — "
            "THETA/PHI here disagree with the parameter master the jobs copied")
        has_unc = unc is not None and np.isfinite(np.asarray(unc)).any()
        print(f"  aperture={i}: OK  flux shape={np.shape(flx)}  MC uncertainties={'yes' if has_unc else 'NO'}")
        n_ok += 1
    except Exception as e:
        print(f"  aperture={i}: FAILED ({type(e).__name__}: {e})")
assert n_ok == N_AP, (
    f"only {n_ok}/{N_AP} apertures readable — the powderday aperture patch is NOT active "
    "(or N_AP here disagrees with SED_APERTURE_NAP in the parameter master the jobs copied)")
print(f"\nOK — {N_AP} apertures present.")

# Part 7 — Stage 2: per-aperture flux extraction → catalogs

For each dust run × aperture: convolve the SED (and its Monte-Carlo uncertainty) with the filter
set, then join the sample metadata. **One catalog per aperture per dust mode** under
`output/cis25/sed_aperture_catalogs/`, columns: `gal_id, snap, redshift`, sample metadata
(`agn_class, log_mstar, ngas, t_sft, t_qt, tau_q, xstr_quench`, …) and per-filter
`<filter>` / `<filter>_err` fluxes (mJy, rest-frame convolution as in `test_powderday.ipynb`).

In [ ]:
# ── filter set (same as test_powderday: optical->FIR so the dust bump is traced) ──
FACILITIES  = ['HST', 'JWST', 'Spitzer', 'Spitzer', 'Herschel', 'Herschel']
INSTRUMENTS = ['WFC3', 'NIRCam', 'IRAC', 'MIPS', 'PACS', 'SPIRE']

local_filters = {
    '2MASS':   {'J': {'J': REMOTE_HOME + '/2MASS_J.res'}},
    'Johnson': {'V': {'V': REMOTE_HOME + '/maiz-apellaniz_Johnson_V.res'}},
    # separate top-level key required: dict cannot hold two entries under 'Johnson'
    'Johnson2': {'U': {'U': REMOTE_HOME + '/maiz-apellaniz_Johnson_U.res'}},
}

SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)

FLUX_FILES = {}
for key in RUNS:
    ms = makeseds[key]
    for i, label in zip(WANTED_AP_IDX, APERTURE_LABELS):
        for j, ilab in enumerate(INCL_LABELS):
            print(f"\n=== extract: {key} / {label} / {ilab} (aperture {i}, inclination {j}) ===")
            flux_file, xmean_file = ms.extract_flux_batch(
                SNAPS, IDS, FACILITIES, INSTRUMENTS,
                filters=None, local_filters=local_filters, wave_unit='micron', findx=j,
                aperture=i, uncertainties=True,
                outname=f"fluxes_{key}_{label}_{ilab}.fits",
            )
            FLUX_FILES[(key, label, ilab)] = flux_file

In [ ]:
# ── final catalogs: fluxes+errors ⨝ sample metadata; one file per (dust mode, aperture) ──
_META = ["gal_id", "snap", "z_snap", "z_target", "agn_class", "xstr_quench",
         "log_mstar", "ngas", "nstar", "ssfr", "t_sft", "t_qt", "tau_q", "tau_q_over_tH",
         "r50_star_kpc", "flag_too_large", "flag_unresolved"]
SEL = Table.read(SELECTION_FITS)
_META = [c for c in _META if c in SEL.colnames]   # size-QC columns exist after Part 3b

CATALOGS = {}
for (key, label, ilab), ff in FLUX_FILES.items():
    t = Table.read(ff)
    if len(t) == 0:
        print(f"[{key}/{label}/{ilab}] EMPTY flux table — skipped"); continue
    t.rename_column('gal_id_at_snap', 'gal_id')
    cat = join(t, SEL[_META], keys=['snap', 'gal_id'], join_type='left')
    flux_cols = [c for c in t.colnames if c not in ('gal_id', 'snap', 'redshift')]
    cat = cat[['gal_id', 'snap', 'redshift'] + [c for c in _META if c not in ('gal_id', 'snap')]
              + flux_cols]
    out = os.path.join(CATDIR, f"catalog_{key}_{label}_{ilab}.fits")
    _k = APERTURE_LABELS.index(label)
    cat.meta['APERTURE'] = label
    cat.meta['APIDX'] = WANTED_AP_IDX[_k]
    cat.meta['APKPC'] = float(APERTURE_RADII_KPC[WANTED_AP_IDX[_k]])   # true rung radius
    cat.meta['INCL'] = ilab
    cat.meta['THETA'] = THETA_DEG[INCL_LABELS.index(ilab)]
    cat.meta['PHI'] = PHI_DEG[INCL_LABELS.index(ilab)]
    cat.meta['DUSTRUN'] = key
    cat.write(out, overwrite=True)
    CATALOGS[(key, label, ilab)] = out
    n_err = sum(1 for c in cat.colnames if c.endswith('_err'))
    print(f"[{key}/{label}/{ilab}] {len(cat)} galaxies, {n_err} error columns -> {out}")

# ── cross-check: per aperture, dust_on and dust_off must contain the same sources ──
print()
for label in APERTURE_LABELS:
    for ilab in INCL_LABELS:
        fon  = FLUX_FILES.get(('dust_on', label, ilab))
        foff = FLUX_FILES.get(('dust_off', label, ilab))
        if fon is None or foff is None:
            continue
        t_on, t_off = Table.read(fon), Table.read(foff)
        s_on  = set(zip(np.asarray(t_on['snap'], int), np.asarray(t_on['gal_id_at_snap'], int)))
        s_off = set(zip(np.asarray(t_off['snap'], int), np.asarray(t_off['gal_id_at_snap'], int)))
        status = "OK" if s_on == s_off else f"MISMATCH on={sorted(s_on - s_off)} off={sorted(s_off - s_on)}"
        print(f"{label:>10s}/{ilab}: dust_on={len(s_on)} dust_off={len(s_off)} -> {status}")

# Part 7a′ — Dust attenuation $A_V$ from the matched dust_on / dust_off fluxes

We already have dust_on **and** dust_off fluxes for the *same* galaxies, so the rest-frame
band attenuation is a direct differential measurement — no SED fit needed:

$$A_\lambda = -2.5\,\log_{10}\!\left(\frac{F_{\rm dust\_on}}{F_{\rm dust\_off}}\right)\ \ [\mathrm{mag}]$$

measured per aperture from the Part-7 catalogs (rest-frame `Johnson.V.V` for $A_V$, plus
`Johnson2.U.U`/`2MASS.J.J` for the curve slope $A_U\!-\!A_V$). The **radial** attenuation
is built the observational way: annular fluxes $F(<r_{\rm out})-F(<r_{\rm in})$ between
consecutive aperture rungs give $A_V$ per annulus (annuli whose differential flux goes
non-positive from MC noise are masked).

$A_V$ is correlated against

- the **anchor-epoch ISM**: $f_{\rm mol}$, $M_{H_2}/M_\star$, $f_{\rm gas}$, dust-to-gas and
  $f_{\rm dust}=M_{\rm dust}/M_\star$ (histories, row 0), plus $\kappa_{\rm rot}$ of the H$_2$
  gas disk — the H$_2$-mass-weighted Sales+2012 $\kappa_{\rm rot}$ inside 20 pkpc, computed
  from the Stage-0 region cutouts (caesar's all-gas $\kappa_{\rm rot}$ kept for comparison);
- the **quench diagnostics + stellar structure at the observation epoch**: sSFR, $M_\star$,
  mass-weighted age, $\log Z_\star/Z_\odot$, stellar $B/T$ (caesar `rotation.stellar_BoverT`),
  $\tau_q$, $\tau_q/t_H$ and the AGN class.

Outputs: `tables/attenuation_vs_ism.fits` (per-galaxy $A_\lambda$ + ISM + structure +
quench/AGN, with $A_V$ per aperture **and** per annulus), a Spearman-ranked correlation
table, and three figures (`attenuation_vs_ism.png`, `attenuation_vs_quench.png`,
`attenuation_aperture_curve.png`).

*Needs Parts 0–3 (histories under `SFHDIR`), the Part-4 region cutouts (for
$\kappa_{\rm rot}^{H_2}$), the anchor caesar catalogs (for $Z$, $B/T$, $\kappa_{\rm rot}$)
and Part 7 (flux catalogs); CIGALE is **not** required.*

In [ ]:
# ── Part 7a′ — Dust attenuation (A_λ) from dust_on/dust_off vs ISM & quenching ──
# Since we already have matched dust_on / dust_off fluxes for the SAME galaxies,
# the rest-frame band attenuation follows directly (no SED fit needed):
#       A_λ = -2.5 log10( F_dust_on / F_dust_off )   [mag]
# measured per aperture from the Part-7 catalogs. The RADIAL profile is built the
# observational way: annular fluxes F(<r_out) - F(<r_in) between consecutive
# aperture rungs -> A_V per annulus. A_V is then correlated against the
# anchor-epoch ISM content (H2/HI/gas/dust from the histories + kappa_rot of the
# H2 disk from the Stage-0 region cutouts) and the quench diagnostics + stellar
# structure (mass, age, metallicity, B/T) at the observation epoch.
from scipy.stats import spearmanr

ATTEN_BANDS = {"A_U": "Johnson2.U.U", "A_V": "Johnson.V.V", "A_J": "2MASS.J.J"}
AGN_COLORS  = {"strong": "#c0392b", "intermediate": "#e67e22", "weak": "#2980b9",
               "no_AGN": "#27ae60", "no_event": "#7f8c8d", "unclassified": "#bdc3c7"}
ZSUN       = 0.0134     # Asplund+2009 total-Z scale (SIMBA's Solar reference)
R_KROT_KPC = 20.0       # H2-disk kappa_rot measured inside this proper radius

def _atten(fon, foff):
    fon = np.asarray(fon, float); foff = np.asarray(foff, float)
    with np.errstate(all="ignore"):
        a = -2.5 * np.log10(fon / foff)
    a[(fon <= 0) | (foff <= 0) | ~np.isfinite(fon) | ~np.isfinite(foff)] = np.nan
    return a

# ── 1. anchor-epoch ISM masses + stellar age (row 0 of each history) ──
GAS_KEYS = ["masses.H2", "masses.HI", "masses.gas", "masses.dust", "masses.stellar",
            "ages.mass_weighted"]
_gasdb = {}
_hist_files = sorted(glob.glob(os.path.join(SFHDIR, "history_anchor_*.hdf5")))
for _hf in _hist_files:
    with h5py.File(_hf, "r") as f:
        _snap0 = int(f["metadata/snapshots"][0])                 # row 0 = anchor epoch
        _gid   = np.asarray(f["metadata/galaxy_ids"][:], int)
        _avail = [k for k in GAS_KEYS if f"properties/{k}" in f]
        _row0  = {k: f[f"properties/{k}"][0] for k in _avail}
    for j, gg in enumerate(_gid):
        _gasdb[(_snap0, int(gg))] = {k: float(_row0[k][j]) for k in _avail}
_missing_gas = [k for k in GAS_KEYS if not any(k in v for v in _gasdb.values())]
if _missing_gas:
    print("WARNING: ISM fields absent from histories (dropped at build):", _missing_gas)
print(f"ISM masses: {len(_gasdb)} (snap,gal) rows from {len(_hist_files)} anchor histories")

def _gp_arr(tab, key):
    """anchor-epoch mass `key` aligned to a catalog table's (snap, gal_id) rows."""
    return np.array([_gasdb.get((int(s), int(g)), {}).get(key, np.nan)
                     for s, g in zip(tab["snap"], tab["gal_id"])])

# ── 1b. anchor-epoch structure from the caesar catalogs (direct h5py read) ──
# metallicities / stellar B/T / gas kappa_rot are not tracked in the histories;
# GroupID == row index in the caesar files, so a plain dataset read suffices.
STRUCT_KEYS = {"Z_star": "metallicities.stellar", "Z_gas": "metallicities.mass_weighted",
               "BT_star": "rotation.stellar_BoverT", "kappa_gas": "rotation.gas_kappa_rot"}
_structdb, _posdb = {}, {}            # (snap,gid) -> {props} / (pos[kpccm], a, h)
_need = {}
for _s, _g in _gasdb:
    _need.setdefault(_s, set()).add(_g)
for _snap, _gids in sorted(_need.items()):
    _cf = sim.get_caesar_file(_snap)
    if not os.path.exists(_cf):
        print(f"WARNING: no caesar file for snap {_snap} -> structure props stay NaN")
        continue
    with h5py.File(_cf, "r") as f:
        _dcts = f["galaxy_data/dicts"]
        _vals = {k: _dcts[v][:] for k, v in STRUCT_KEYS.items() if v in _dcts}
        _pos  = f["galaxy_data/pos"][:]                          # kpccm
        _sa   = f["simulation_attributes"].attrs
        _a, _h = float(_sa["scale_factor"]), float(_sa["hubble_constant"])
    _absent = [v for k, v in STRUCT_KEYS.items() if k not in _vals]
    if _absent:
        print(f"WARNING: snap {_snap} caesar file lacks {_absent}")
    for _g in _gids:
        if _g < len(_pos):
            _structdb[(_snap, _g)] = {k: float(_arr[_g]) for k, _arr in _vals.items()}
            _posdb[(_snap, _g)]    = (np.asarray(_pos[_g], float), _a, _h)
print(f"structure props: {len(_structdb)} (snap,gal) rows from the anchor caesar catalogs")

def _sp_arr(tab, key):
    """anchor-epoch structure prop `key` aligned to a catalog table's rows."""
    return np.array([_structdb.get((int(s), int(g)), {}).get(key, np.nan)
                     for s, g in zip(tab["snap"], tab["gal_id"])])

# ── 1c. kappa_rot of the H2 gas disk from the Stage-0 region cutouts ──
def _kappa_rot_h2(snap, gid):
    """Sales+12 kappa_rot of the H2-mass-weighted gas within R_KROT_KPC (proper).

    Cutout Coordinates are code units (ckpc/h) unwrapped around the caesar centre,
    so pos_kpccm*h recovers that centre exactly; the uniform sqrt(a) factor of the
    code velocities cancels in the K_rot/K ratio.
    """
    rec = _posdb.get((int(snap), int(gid)))
    pf  = os.path.join(hydro_dir_base, f"snap_{int(snap):03d}",
                       f"{PARTICLE_PREFIX}_snap{int(snap):03d}_gal{int(gid):06d}.h5")
    if rec is None or not os.path.exists(pf):
        return np.nan
    pos_kpccm, a_scale, hub = rec
    with h5py.File(pf, "r") as f:
        if "PartType0" not in f:
            return np.nan
        g   = f["PartType0"]
        x   = g["Coordinates"][:].astype(float)
        v   = g["Velocities"][:].astype(float)
        mh2 = g["Masses"][:].astype(float) * g["FractionH2"][:].astype(float)
    r   = (x - pos_kpccm * hub) * (a_scale / hub)               # proper kpc, gal frame
    sel = (np.sqrt(np.sum(r**2, axis=1)) < R_KROT_KPC) & (mh2 > 0) & np.isfinite(mh2)
    if sel.sum() < 10:
        return np.nan
    r, vv, w = r[sel], v[sel], mh2[sel]
    r  = r  - np.average(r,  axis=0, weights=w)                 # recentre on the H2 body
    vv = vv - np.average(vv, axis=0, weights=w)
    j  = np.cross(r, vv)
    L  = np.sum(w[:, None] * j, axis=0)
    if not np.isfinite(L).all() or np.linalg.norm(L) == 0:
        return np.nan
    zhat = L / np.linalg.norm(L)
    jz   = j @ zhat
    Rcyl = np.sqrt(np.maximum(np.sum(r**2, axis=1) - (r @ zhat)**2, 0.0))
    ok   = Rcyl > 1e-3
    Krot = 0.5 * np.sum(w[ok] * (jz[ok] / Rcyl[ok])**2)
    Ktot = 0.5 * np.sum(w * np.sum(vv**2, axis=1))
    return float(Krot / Ktot) if Ktot > 0 else np.nan

# ── 2. per-aperture attenuation table (dust_on ⨝ dust_off on snap+gal_id) ──
ATTEN_INCL = INCL_LABELS[0]     # fiducial sightline for the A_λ analysis
def _load_atten(label, incl=None):
    incl = ATTEN_INCL if incl is None else incl
    fon  = os.path.join(CATDIR, f"catalog_dust_on_{label}_{incl}.fits")
    foff = os.path.join(CATDIR, f"catalog_dust_off_{label}_{incl}.fits")
    if not (os.path.exists(fon) and os.path.exists(foff)):
        return None
    on, off = Table.read(fon), Table.read(foff)
    off_cols = ["snap", "gal_id"] + [c for c in ATTEN_BANDS.values() if c in off.colnames]
    m = join(on, off[off_cols], keys=["snap", "gal_id"],
             table_names=["on", "off"], metadata_conflicts="silent")
    for aname, col in ATTEN_BANDS.items():
        m[aname] = (_atten(m[f"{col}_on"], m[f"{col}_off"])
                    if f"{col}_on" in m.colnames and f"{col}_off" in m.colnames
                    else np.full(len(m), np.nan))
    return m

_atab = {lab: _load_atten(lab) for lab in APERTURE_LABELS}
_atab = {k: v for k, v in _atab.items() if v is not None and len(v)}
if not _atab:
    raise FileNotFoundError(f"no dust_on/dust_off catalogs in {CATDIR}; run Part 7 first")
FID_AP = APERTURE_LABELS[-1] if APERTURE_LABELS[-1] in _atab else list(_atab)[-1]
print(f"apertures with catalogs: {list(_atab)}  |  fiducial (global A_V) = {FID_AP}"
      f"  |  sightline = {ATTEN_INCL}")

# ── 3. fiducial-aperture analysis table: A_λ + ISM tracers + structure + quench ──
base  = _atab[FID_AP]
_MH2  = _gp_arr(base, "masses.H2");   _MHI = _gp_arr(base, "masses.HI")
_Mgas = _gp_arr(base, "masses.gas");  _Md  = _gp_arr(base, "masses.dust")
_Mst  = _gp_arr(base, "masses.stellar")
with np.errstate(all="ignore"):
    f_mol       = _MH2 / (_MH2 + _MHI)          # molecular fraction of neutral gas
    f_H2_star   = _MH2 / _Mst                    # specific molecular content
    f_gas       = _Mgas / (_Mgas + _Mst)         # gas fraction
    DGR         = _Md / _Mgas                     # dust-to-gas ratio
    f_dust_star = _Md / _Mst                      # f_dust: specific dust content

ATTEN = Table()
ATTEN["snap"]     = np.asarray(base["snap"], int)
ATTEN["gal_id"]   = np.asarray(base["gal_id"], int)
ATTEN["z_target"] = np.asarray(base["z_target"], float)
for aname in ATTEN_BANDS:
    ATTEN[aname] = np.asarray(base[aname], float)
ATTEN["S_UV"] = ATTEN["A_U"] - ATTEN["A_V"]      # attenuation-curve slope proxy (mag)
for lab, tt in _atab.items():                    # enclosed A_V at every aperture
    idx = {(int(s), int(g)): k for k, (s, g) in enumerate(zip(tt["snap"], tt["gal_id"]))}
    col = np.full(len(base), np.nan)
    for k, (s, g) in enumerate(zip(base["snap"], base["gal_id"])):
        j = idx.get((int(s), int(g)))
        if j is not None:
            col[k] = tt["A_V"][j]
    ATTEN[f"A_V_{lab}"] = col
with np.errstate(all="ignore"):
    ATTEN["log_MH2"]   = np.log10(np.where(_MH2 > 0, _MH2, np.nan))
    ATTEN["log_Mgas"]  = np.log10(np.where(_Mgas > 0, _Mgas, np.nan))
    ATTEN["log_Mdust"] = np.log10(np.where(_Md > 0, _Md, np.nan))
ATTEN["f_mol"] = f_mol; ATTEN["f_H2_star"] = f_H2_star; ATTEN["f_gas"] = f_gas
ATTEN["DGR"] = DGR; ATTEN["f_dust_star"] = f_dust_star
for c in ("log_mstar", "ssfr", "tau_q", "tau_q_over_tH", "xstr_quench"):
    ATTEN[c] = np.asarray(base[c], float)
_agn = np.asarray(base["agn_class"])
ATTEN["agn_class"] = np.array([a.decode() if isinstance(a, (bytes, np.bytes_)) else str(a)
                               for a in _agn])

# anchor-epoch stellar structure + gas-disk rotation (observation time)
ATTEN["age_mw"] = _gp_arr(base, "ages.mass_weighted")          # Gyr, mass-weighted
with np.errstate(all="ignore"):
    ATTEN["logZ_star"] = np.log10(_sp_arr(base, "Z_star") / ZSUN)
    ATTEN["logZ_gas"]  = np.log10(_sp_arr(base, "Z_gas") / ZSUN)
ATTEN["BT_star"]   = _sp_arr(base, "BT_star")
ATTEN["kappa_gas"] = _sp_arr(base, "kappa_gas")
ATTEN["kappa_H2"]  = np.array([_kappa_rot_h2(s, g)
                               for s, g in zip(base["snap"], base["gal_id"])])
print(f"kappa_H2 (<{R_KROT_KPC:g} pkpc): measured for "
      f"{int(np.isfinite(np.asarray(ATTEN['kappa_H2'])).sum())}/{len(ATTEN)} galaxies "
      f"(needs the Stage-0 cutouts + >=10 H2-bearing gas particles)")

# ── 3b. annular A_V — the observational radial profile: F(<r_out) - F(<r_in) ──
_labs_all = [l for l in APERTURE_LABELS if l in _atab]
_r_out    = np.array([APERTURE_RADII_KPC[WANTED_AP_IDX[APERTURE_LABELS.index(l)]]
                      for l in _labs_all])                     # TRUE rung radii [pkpc]
_r_in     = np.concatenate([[0.0], _r_out[:-1]])
_r_mid    = np.where(_r_in > 0, np.sqrt(_r_in * _r_out), _r_out / 2.0)

def _band_matrix(col):
    """(n_gal, n_ap) matrix of catalog column `col`, aligned to the base rows."""
    M = np.full((len(base), len(_labs_all)), np.nan)
    for k, lab in enumerate(_labs_all):
        tt = _atab[lab]
        if col not in tt.colnames:
            continue
        idx = {(int(s), int(g)): j for j, (s, g) in enumerate(zip(tt["snap"], tt["gal_id"]))}
        for i, (s, g) in enumerate(zip(base["snap"], base["gal_id"])):
            j = idx.get((int(s), int(g)))
            if j is not None:
                M[i, k] = tt[col][j]
    return M

_Von, _Voff = _band_matrix("Johnson.V.V_on"), _band_matrix("Johnson.V.V_off")
_dVon  = np.column_stack([_Von[:, :1],  np.diff(_Von,  axis=1)])   # annular fluxes
_dVoff = np.column_stack([_Voff[:, :1], np.diff(_Voff, axis=1)])
AV_ANN = _atten(_dVon, _dVoff)
for k, lab in enumerate(_labs_all):
    ATTEN[f"A_V_ann_{lab}"] = AV_ANN[:, k]
_nneg = int(np.sum(((_dVon <= 0) | (_dVoff <= 0)) & np.isfinite(_Von) & np.isfinite(_Voff)))
print(f"annular A_V: {len(_labs_all)} annuli/galaxy; {_nneg} annuli with non-positive "
      f"differential flux (MC noise / empty annulus) -> NaN")

_out = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
ATTEN.write(_out, overwrite=True)
_av = np.asarray(ATTEN["A_V"], float)
print(f"attenuation table: {len(ATTEN)} galaxies ({FID_AP}) -> {_out}")
print(f"A_V [{FID_AP}]  median={np.nanmedian(_av):.3f}  90th={np.nanpercentile(_av,90):.3f}  "
      f"max={np.nanmax(_av):.3f}   (A_V>0.1 mag: {int(np.nansum(_av>0.1))}/{len(ATTEN)})")

# ── 4. Spearman correlations of the global A_V with everything ──
_targets = [("f_mol","f_mol"), ("f_H2_star","M_H2/M*"), ("f_gas","f_gas"),
            ("DGR","dust/gas"), ("f_dust_star","f_dust"), ("log_MH2","log M_H2"),
            ("log_Mgas","log M_gas"), ("kappa_H2","kappa_H2"), ("kappa_gas","kappa_gas"),
            ("log_mstar","log M*"), ("age_mw","age_mw"), ("logZ_star","log Z*/Zsun"),
            ("logZ_gas","log Zg/Zsun"), ("BT_star","B/T"), ("ssfr","sSFR"),
            ("tau_q","tau_q"), ("tau_q_over_tH","tau_q/t_H"), ("S_UV","A_U-A_V")]
_ranked = []
for col, lbl in _targets:
    x = np.asarray(ATTEN[col], float); ok = np.isfinite(_av) & np.isfinite(x)
    if ok.sum() >= 5:
        rho, p = spearmanr(_av[ok], x[ok]); _ranked.append((lbl, rho, p, int(ok.sum())))
_ranked.sort(key=lambda r: -abs(r[1]))
print("\nSpearman  A_V  vs …   (fiducial aperture, sorted by |rho|)")
print(f"  {'quantity':12s} {'rho':>7s} {'p':>10s} {'n':>4s}")
for lbl, rho, p, n in _ranked:
    flag = "***" if p < 0.01 else "** " if p < 0.05 else "*  " if p < 0.1 else ""
    print(f"  {lbl:12s} {rho:+7.3f} {p:10.2e} {n:4d}  {flag}")

# per-aperture robustness of the two headline ISM correlations
print("\nrobustness across apertures  (rho[p]):")
for lab in _labs_all:
    tt = _atab[lab]; av = np.asarray(tt["A_V"], float)
    with np.errstate(all="ignore"):
        fh2 = _gp_arr(tt, "masses.H2") / _gp_arr(tt, "masses.stellar")
        dgr = _gp_arr(tt, "masses.dust") / _gp_arr(tt, "masses.gas")
    def _rp(x):
        ok = np.isfinite(av) & np.isfinite(x)
        return spearmanr(av[ok], x[ok]) if ok.sum() >= 5 else (np.nan, np.nan)
    (r1, p1), (r2, p2) = _rp(fh2), _rp(dgr)
    print(f"  {lab:16s}  M_H2/M*: {r1:+.2f}[{p1:.2g}]   dust/gas: {r2:+.2f}[{p2:.2g}]")

# ── 5. figures ──
_cls_present = [c for c in ["strong","intermediate","weak","no_AGN","no_event","unclassified"]
                if c in set(ATTEN["agn_class"])]
def _scatter(ax, xcol, xlabel, xlog=False):
    x = np.asarray(ATTEN[xcol], float); y = _av
    for cls in _cls_present:
        s = ATTEN["agn_class"] == cls
        ax.scatter(x[s], y[s], s=28, c=AGN_COLORS.get(cls, "#333"), label=cls,
                   edgecolor="k", linewidth=0.3, alpha=0.85)
    ok = np.isfinite(x) & np.isfinite(y)
    if xlog: ok &= x > 0
    if ok.sum() >= 5:
        rho, p = spearmanr(x[ok], y[ok])
        ax.text(0.04, 0.95, f"$\\rho$={rho:+.2f}\np={p:.2g}", transform=ax.transAxes,
                va="top", fontsize=8.5, bbox=dict(fc="white", ec="0.7", alpha=0.85, pad=1.6))
    if xlog:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel(r"$A_V$ [mag]")

# Fig 1 — attenuation vs ISM / dust content (the H2 connection)
_p1 = [("f_mol", r"$f_{\rm mol}=M_{H_2}/(M_{H_2}\!+\!M_{HI})$", False),
       ("f_H2_star", r"$M_{H_2}/M_\star$", True),
       ("f_gas", r"$f_{\rm gas}=M_{\rm gas}/(M_{\rm gas}\!+\!M_\star)$", False),
       ("DGR", r"dust-to-gas $M_{\rm dust}/M_{\rm gas}$", True),
       ("f_dust_star", r"$f_{\rm dust}=M_{\rm dust}/M_\star$", True),
       ("kappa_H2", r"$\kappa_{\rm rot}^{H_2}$ ($<$%g pkpc)" % R_KROT_KPC, False)]
fig, axes = plt.subplots(2, 3, figsize=(15, 8.6))
for ax, (c, xl, xlog) in zip(axes.flat, _p1):
    _scatter(ax, c, xl, xlog)
axes.flat[0].legend(fontsize=7.5, loc="upper right", framealpha=0.9)
fig.suptitle(f"Rest-frame $A_V$ (dust_on/dust_off, {FID_AP}) vs anchor-epoch ISM content",
             fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
_f1 = os.path.join(PLOTDIR, "attenuation_vs_ism.png")
fig.savefig(_f1, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f1)

# Fig 2 — attenuation vs quenching + stellar structure at the observation epoch
_p2 = [("ssfr", "sSFR [yr$^{-1}$]", True),
       ("log_mstar", r"$\log_{10} M_\star\,[M_\odot]$", False),
       ("age_mw", "mass-weighted age [Gyr]", False),
       ("logZ_star", r"$\log_{10} Z_\star/Z_\odot$", False),
       ("BT_star", r"stellar $B/T$", False),
       ("tau_q", r"$\tau_q$ [yr]", False),
       ("tau_q_over_tH", r"$\tau_q/t_H$", False)]
fig, axes = plt.subplots(2, 4, figsize=(18.5, 8.4))
for ax, (c, xl, xlog) in zip(axes.flat[:7], _p2):
    _scatter(ax, c, xl, xlog)
ax = axes.flat[7]
for i, cls in enumerate(_cls_present):
    s = ATTEN["agn_class"] == cls; yv = _av[np.asarray(s)]
    xj = i + np.random.uniform(-0.16, 0.16, size=int(np.sum(s)))
    ax.scatter(xj, yv, c=AGN_COLORS.get(cls, "#333"), edgecolor="k", linewidth=0.3, s=28)
    if np.isfinite(yv).any():
        ax.hlines(np.nanmedian(yv), i - 0.3, i + 0.3, color="k", lw=2)
ax.set_xticks(range(len(_cls_present)))
ax.set_xticklabels(_cls_present, rotation=30, ha="right", fontsize=8.5)
ax.set_ylabel(r"$A_V$ [mag]"); ax.set_title("by AGN class", fontsize=9)
fig.suptitle(f"Rest-frame $A_V$ ({FID_AP}) vs quenching & stellar structure "
             f"at the observation epoch", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
_f2 = os.path.join(PLOTDIR, "attenuation_vs_quench.png")
fig.savefig(_f2, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f2)

# Fig 3 — radial A_V from annular fluxes + attenuation-curve slope
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.8))
_AVap  = np.vstack([np.asarray(ATTEN[f"A_V_{l}"], float) for l in _labs_all]).T
_AVann = np.vstack([np.asarray(ATTEN[f"A_V_ann_{l}"], float) for l in _labs_all]).T
_dusty = _av > 0.1                                   # most quenched gals are transparent;
for row in _AVann[_dusty]:                           # the radial trend only matters there
    axL.plot(_r_mid, row, "-", color="0.7", lw=0.8, alpha=0.7, zorder=1)
axL.plot(_r_mid, np.nanmedian(_AVann[_dusty], axis=0), "o-", color="#c0392b", lw=2,
         label=f"annular median, $A_V\\!>\\!0.1$ (n={int(_dusty.sum())})", zorder=3)
axL.plot(_r_mid, np.nanmedian(_AVann, axis=0), "s--", color="#2980b9",
         label=f"annular median, all (n={len(_av)})", zorder=2)
axL.plot(_r_out, np.nanmedian(_AVap[_dusty], axis=0), ":", color="0.35", lw=1.5,
         label=r"enclosed $A_V(<r)$, $A_V\!>\!0.1$", zorder=2)
axL.set_xscale("log"); axL.set_xlabel("radius [pkpc]")
axL.set_ylabel(r"$A_V$ [mag]")
axL.set_title(r"radial $A_V$: annuli $F(<r_{\rm out})-F(<r_{\rm in})$")
axL.legend(fontsize=8)
for cls in _cls_present:
    s = ATTEN["agn_class"] == cls
    axR.scatter(_av[np.asarray(s)], np.asarray(ATTEN["S_UV"])[np.asarray(s)], s=28,
                c=AGN_COLORS.get(cls, "#333"), edgecolor="k", linewidth=0.3, alpha=0.85, label=cls)
axR.axhline(0, color="0.6", lw=0.8, ls="--")
axR.set_xlabel(r"$A_V$ [mag]"); axR.set_ylabel(r"$A_U-A_V$ [mag]  (curve slope)")
axR.set_title("attenuation depth vs UV-optical reddening"); axR.legend(fontsize=7.5)
fig.tight_layout()
_f3 = os.path.join(PLOTDIR, "attenuation_aperture_curve.png")
fig.savefig(_f3, dpi=130, bbox_inches="tight"); plt.show()
print("saved", _f3)

# Part 7b — CIGALE input files (one per dust mode × aperture)

Writes `output/cis25/sed_aperture_catalogs/cigale/cigale_{dust_on|dust_off}_{ap…}.fits` in the
exact input format of **CIGALE 2025.0**. All the format/mapping logic lives in
**`simbanator.sed.cigale`** (band names verified against the 2025.0 filter database):

- columns `id` (`snapNNN_galID`), `redshift`, `distance` (Mpc, Planck13 — the same D_L used to
  normalize the fluxes), then per band the flux **in mJy** + its `<band>_err`;
- band names match the CIGALE DB exactly (`jwst.nircam.F200W`, `hst.wfc3.ir.F160W`,
  `spitzer.irac.I1`, `herschel.pacs.green`, `2mass.J`, `generic.johnson.U/V`, …); bands with no
  CIGALE counterpart (grisms, quad filters) are dropped and reported;
- missing fluxes are NaN; make an error negative by hand for upper-limit treatment.

Two deliberate choices:

1. **Observed frame.** CIGALE compares redshifted models to observed photometry, so the
   extraction reruns with `redshift=True` (the Part 7 catalogs stay rest-frame).
2. **Raw MC errors (`err_floor=0`).** CIGALE itself adds `additionalerror` (10 % by default,
   set in Part 7c's `prepare_run`) in quadrature at fit time — a floor here too would be
   double-counted. The Hyperion MC error alone is just RT convergence noise.

In [ ]:
# ── Part 7b: CIGALE 2025.0 input files — observed-frame fluxes+errors, one per (dust mode, aperture) ──
# Format + band mapping live in simbanator.sed.cigale (verified against the 2025.0 filter DB).
# Needs the Part 5 MakeSED handles + the Part 7 filter-set cell in this session.
from simbanator.sed.cigale import write_cigale_input

CIGALE_DIR = os.path.join(CATDIR, "cigale")
os.makedirs(CIGALE_DIR, exist_ok=True)

SEL = Table.read(SELECTION_FITS)
SNAPS = np.asarray(SEL["snap"], int)
IDS   = np.asarray(SEL["gal_id"], int)

CIGALE_FILES = {}
for key in RUNS:
    ms = makeseds[key]
    for i, label in zip(WANTED_AP_IDX, APERTURE_LABELS):
        for j, ilab in enumerate(INCL_LABELS):
            print(f"\n=== CIGALE extract: {key} / {label} / {ilab} (aperture {i}, observed frame) ===")
            flux_file, _ = ms.extract_flux_batch(
                SNAPS, IDS, FACILITIES, INSTRUMENTS,
                filters=None, local_filters=local_filters, wave_unit='micron', findx=j,
                aperture=i, uncertainties=True,
                redshift=True,                # CIGALE wants observed-frame fluxes at 'redshift'
                outname=f"cigale_fluxes_{key}_{label}_{ilab}.fits",
            )
            # err_floor=0: CIGALE adds its own 10% 'additionalerror' in quadrature at fit time
            CIGALE_FILES[(key, label, ilab)] = write_cigale_input(
                flux_file, os.path.join(CIGALE_DIR, f"cigale_{key}_{label}_{ilab}.fits"),
                err_floor=0.0)

print(f"\n{len(CIGALE_FILES)} CIGALE input files -> {CIGALE_DIR}")

# Part 7b″ — annular CIGALE inputs: $F(<r_{\rm out}) - F(<r_{\rm in})$ between consecutive rungs

The Part 7b catalogs are **cumulative** (each aperture contains all the inner light), so any
radial trend fitted from them is a curve-of-growth. This part builds true **annular** catalogs
the observational way — differencing the observed-frame per-aperture fluxes band by band
(`simbanator.sed.flux_extraction.annular_flux_table`), the same construction as Part 7a′'s
annular $A_V$. As there, the **outer** rung names the annulus: `ann3kpc` = 1→3.16 kpc, …,
`ann100kpc` = 31.6→100 kpc. `ann1kpc` (0→1 kpc) **is** the `ap1kpc` aperture and is not
duplicated — the radial set downstream is `ap1kpc + ann3kpc…ann100kpc`.

- **Non-positive annular flux** (MC noise / empty annulus) → NaN flux+error = missing band to
  CIGALE. Objects losing most bands this way will fit poorly — watch the printed counts.
- **Errors**: `sqrt(err_out² − err_in²)` — the photons inside $r_{\rm in}$ are counted in both
  rungs, so the cumulative variances subtract; where MC noise makes the difference
  non-positive, the conservative quadrature **sum** is used instead.
- **Sampling QC**: with Part 4b's `annulus_particle_counts.fits` present, sources with **zero
  star particles** in an annulus are listed per catalog — their annular flux is scattered
  light only and the CIGALE fit is meaningless (drop them via the QC table when interpreting).
- Files land next to the Part 7b ones (`cigale/cigale_{dust}_{ann…}_{incl}.fits`), so
  **Part 7c picks them up automatically** (+32 runs = 2 dust modes × 4 annuli × 4 sightlines).
  Part 7d skips `_ann` tags: the global truth (total $M_*$, SFR) is not per-annulus.

Needs the Part 5 MakeSED-handles cell; the Part 7b per-aperture
`cigale_fluxes_*.fits` intermediates are read from disk (run Part 7b once first).

In [ ]:
# ── Part 7b″: annular CIGALE inputs — F(<r_out) − F(<r_in) between consecutive rungs ──
from simbanator.sed.flux_extraction import annular_flux_table
from simbanator.sed.cigale import write_cigale_input

CIGALE_DIR = os.path.join(CATDIR, "cigale")
os.makedirs(CIGALE_DIR, exist_ok=True)
ANN_LABELS = [l.replace("ap", "ann") for l in APERTURE_LABELS]   # outer rung names the annulus

_qcf = os.path.join(TABLEDIR, "annulus_particle_counts.fits")
_qc = Table.read(_qcf) if os.path.exists(_qcf) else None
if _qc is None:
    print("[QC] annulus_particle_counts.fits not found — run Part 4b to flag star-free annuli")

CIGALE_ANN_FILES = {}
for key in RUNS:
    ms = makeseds[key]
    _fluxdir = os.path.join(ms.output_dir, ms.run_tag, 'sed_fluxes')
    for ilab in INCL_LABELS:
        _files = [os.path.join(_fluxdir, f"cigale_fluxes_{key}_{lab}_{ilab}.fits")
                  for lab in APERTURE_LABELS]
        _miss = [os.path.basename(p) for p in _files if not os.path.exists(p)]
        if _miss:
            raise FileNotFoundError(f"{key}/{ilab}: run Part 7b first — missing {_miss}")
        for k in range(1, len(APERTURE_LABELS)):     # k=0 (ann1kpc) == ap1kpc, already fit
            alab = ANN_LABELS[k]
            print(f"\n=== annularize: {key} / {alab} "
                  f"({APERTURE_LABELS[k-1]} -> {APERTURE_LABELS[k]}) / {ilab} ===")
            _tann = annular_flux_table(_files[k - 1], _files[k])
            CIGALE_ANN_FILES[(key, alab, ilab)] = write_cigale_input(
                _tann, os.path.join(CIGALE_DIR, f"cigale_{key}_{alab}_{ilab}.fits"),
                err_floor=0.0)   # CIGALE adds its own 10% additionalerror at fit time
            if _qc is not None and key == 'dust_on':          # QC once per (annulus, incl)
                _m = np.char.strip(np.asarray(_qc["incl"], str)) == ilab
                _z = np.asarray(_qc[f"nstar_{alab}"], int)[_m] == 0
                if _z.any():
                    _who = [f"snap{int(s):03d}_gal{int(g)}"
                            for s, g in zip(np.asarray(_qc["snap"], int)[_m][_z],
                                            np.asarray(_qc["gal_id"], int)[_m][_z])]
                    print(f"    [QC] {int(_z.sum())} source(s) with ZERO star particles in "
                          f"{alab}/{ilab} (annular flux = scattered light only): {_who}")

print(f"\n{len(CIGALE_ANN_FILES)} annular CIGALE input files -> {CIGALE_DIR}")

# Part 7b′ — SFH-derived priors: true parameter ranges & SFH fits

Before running CIGALE, check what the **simulation's own** star-formation histories imply for the fit grid. For each selected galaxy this cell:

- **smooths + resamples** the SFH first (Gaussian kernel ≈ the snapshot cadence → uniform 25 Myr grid; `simbanator.analysis.sfh_utils.smooth_resample_sfh`) — SIMBA's per-snapshot SFR is instantaneous and bursty, so the smooth CIGALE forms are fitted to the smoothed track, while the stochastic burst statistics keep the **raw** one (smoothing would erase exactly what they measure);
- tabulates the true ranges of `M*`, `sSFR`, the quench duration `tau_q`, and the quench lookback `age_bq = t_obs − t_qt`;
- fits the **delayed-τ + burst/quench** form (CIGALE `sfhdelayedbq`) and a **double-exponential** form to `SFR(t)`, giving the empirical priors for `tau_main`, `age_main`, `age_bq`, `r_sfr`;
- measures the **burst statistics** for the Carvajal-Bohórquez et al. (2025) **stochastic** SFH (`sfhstochastic_carvajal2025`) — `sigma` (dex, SFR-variability amplitude) and `tau_break` (Myr, decorrelation time) — the bursts the smooth forms cannot represent;
- prints the p5/p50/p95 of each fitted parameter next to the **current CIGALE grid nodes**, and plots example SFHs + fits, the median stacked SFH, and the fitted-parameter distributions against those nodes.

Use the printed *suggested grid* and the dashed-node histograms to decide where to extend/refine the `sfhdelayedbq` grid in Part 7c. Requires the anchor histories (Part 1, `BUILD_MULTI_Z`) and the selection catalog (Parts 2–3).

In [ ]:
# ── Part 7b′ — Priors from the TRUE SFHs: parameter ranges + SFH fits ──────────
# Self-contained after Part 1 (needs ANCHORS + load_anchor_history) and the
# selection catalog (SELECTION_FITS). From SIMBA's own star-formation histories
# it (1) tabulates the parameter ranges the CIGALE grid should span and (2) fits
# the sfhdelayedbq and a double-exponential form to each galaxy AND (3) measures
# the burst statistics for the Carvajal+2025 stochastic SFH (sigma, tau_break) —
# the bursts the smooth delayed forms miss. Plots example SFHs + fits (with the
# stochastic trend + +/-sigma burst band), the median stacked SFH, and the
# parameter distributions against the current CIGALE nodes.
from scipy.optimize import curve_fit
from simbanator.sed.cigale import DEFAULT_MODULE_PARAMS
from simbanator.analysis.sfh_utils import (smooth_resample_sfh,
                                            sfr_delayed_bq, fit_delayed_bq)

# SIMBA's per-snapshot SFR is instantaneous and bursty: smooth + resample each
# track before fitting the smooth CIGALE forms (delayed+bq, 2-exp). The
# stochastic burst statistics are still measured on the RAW track.
SFH_SMOOTH_KERNEL_MYR = None    # Gaussian sigma [Myr]; None -> median snapshot spacing
SFH_RESAMPLE_DT_MYR   = 25.0    # uniform output grid step [Myr]


def sfr_2exp(t, A1, tau1, A2, tau2, t0):
    """Double-exponential SFH: two declining components from t0 (Gyr)."""
    t = np.asarray(t, float)
    x = np.clip(t - t0, 0.0, None)
    return A1 * np.exp(-x / tau1) + A2 * np.exp(-x / tau2)


def fit_2exp(t, sfr):
    """Bounded fit of sfr_2exp (taus ordered fast<slow); dict (Myr) or None."""
    t = np.asarray(t, float); sfr = np.asarray(sfr, float)
    ok = np.isfinite(t) & np.isfinite(sfr) & (sfr >= 0)
    t, sfr = t[ok], sfr[ok]
    if t.size < 5:
        return None
    o = np.argsort(t); t, sfr = t[o], sfr[o]
    t0 = t[0]; peak = max(sfr.max(), 1e-6)
    p0 = [peak, 0.5, peak * 0.3, 3.0]
    lo = [0.0, 0.05, 0.0, 0.05]; hi = [peak * 1e3, 20.0, peak * 1e3, 20.0]
    try:
        popt, _ = curve_fit(lambda tt, A1, tau1, A2, tau2:
                            sfr_2exp(tt, A1, tau1, A2, tau2, t0),
                            t, sfr, p0=p0, bounds=(lo, hi), maxfev=20000)
    except Exception:
        return None
    A1, tau1, A2, tau2 = popt
    if tau1 > tau2:
        A1, tau1, A2, tau2 = A2, tau2, A1, tau1
    return dict(A1=A1, tau1_myr=tau1 * 1e3, A2=A2, tau2_myr=tau2 * 1e3, t0=t0)


def stochastic_stats(t, sfr, min_points=8, deg=3):
    """Burst statistics for the Carvajal-Bohorquez+2025 stochastic SFH.
    Detrend log10(SFR) over star-forming epochs with a low-order polynomial
    (removes the delayed rise/fall + quench), then sigma [dex] = residual
    scatter, tau_break [Myr] = residual decorrelation time (ACF hits 1/e).
    Note: tau_break cannot resolve below the snapshot cadence dt_myr."""
    t = np.asarray(t, float); sfr = np.asarray(sfr, float)
    ok = np.isfinite(t) & np.isfinite(sfr) & (sfr > 0)
    t, sfr = t[ok], sfr[ok]
    if t.size < min_points:
        return None
    o = np.argsort(t); t, sfr = t[o], sfr[o]
    y = np.log10(sfr)
    d = int(min(deg, t.size - 2))
    coef = np.polyfit(t, y, d)
    resid = y - np.polyval(coef, t)
    sigma = float(np.std(resid))
    dt = float(np.median(np.diff(t)))
    tau_break = np.nan
    if dt > 0:
        tu = np.arange(t[0], t[-1] + 0.5 * dt, dt)
        ru = np.interp(tu, t, resid); ru = ru - ru.mean()
        if ru.size >= 4 and np.any(ru != 0):
            ac = np.correlate(ru, ru, "full")[ru.size - 1:]; ac = ac / ac[0]
            below = np.where(ac < 1.0 / np.e)[0]
            tau_break = float((below[0] if below.size else ru.size) * dt * 1e3)
    return dict(sigma_dex=sigma, tau_break_myr=tau_break, n=int(t.size),
                dt_myr=dt * 1e3, poly=coef, t0=float(t[0]), t1=float(t[-1]))


# ── gather the selected sample's true SFHs (SFR vs cosmic time) ──
SEL = Table.read(SELECTION_FITS)
SFH = []
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        continue
    H = load_anchor_history(A)
    gids = np.asarray(H["galaxy_ids"])
    t_gyr = np.asarray(H["t_cosmic_yr"], float) / 1e9
    sfr_all = np.asarray(H["P"]["sfr"], float)          # (n_snap, n_gal)
    order = np.argsort(t_gyr)
    gid_to_col = {int(g): i for i, g in enumerate(gids)}
    for row in SEL[SEL["snap"] == int(A["snap"])]:
        col = gid_to_col.get(int(row["gal_id"]))
        if col is None:
            continue
        tg = t_gyr[order]; sg = sfr_all[order, col]
        ts, ss = smooth_resample_sfh(tg, sg, dt_myr=SFH_RESAMPLE_DT_MYR,
                                     kernel_myr=SFH_SMOOTH_KERNEL_MYR)
        t_obs = float(tg[-1])
        tqt = float(row["t_qt"]) if np.isfinite(row["t_qt"]) else np.nan
        age_bq_emp = (t_obs * 1e9 - tqt) / 1e6 if np.isfinite(tqt) else np.nan
        SFH.append(dict(
            gal_id=int(row["gal_id"]), snap=int(A["snap"]), z=float(A["z"]),
            t=ts, sfr=ss, t_raw=tg, sfr_raw=sg,
            t_obs=t_obs, log_mstar=float(row["log_mstar"]),
            ssfr=float(row["ssfr"]),
            tau_q_gyr=(float(row["tau_q"]) / 1e9 if np.isfinite(row["tau_q"]) else np.nan),
            age_bq_emp_myr=age_bq_emp))

if not SFH:
    raise RuntimeError("no SFHs to fit — build the anchor histories "
                       "(BUILD_MULTI_Z, Part 1) and the selection catalog "
                       "(Parts 2-3) first")

for g in SFH:
    _ab0 = (g["age_bq_emp_myr"] / 1e3 if np.isfinite(g["age_bq_emp_myr"]) else None)
    g["bq"] = fit_delayed_bq(g["t"], g["sfr"], g["t_obs"], age_bq0=_ab0)  # smoothed
    g["e2"] = fit_2exp(g["t"], g["sfr"])                                   # smoothed
    g["st"] = stochastic_stats(g["t_raw"], g["sfr_raw"])   # bursts: RAW track
n_bq = sum(g["bq"] is not None for g in SFH)
n_e2 = sum(g["e2"] is not None for g in SFH)
n_st = sum(g["st"] is not None for g in SFH)
_kern = ("median-cadence" if SFH_SMOOTH_KERNEL_MYR is None
         else f"{SFH_SMOOTH_KERNEL_MYR:g} Myr")
print(f"[SFH priors] {len(SFH)} selected galaxies | smoothed (kernel={_kern}, "
      f"dt={SFH_RESAMPLE_DT_MYR:g} Myr) | delayedbq fit ok: {n_bq} | "
      f"2exp fit ok: {n_e2} | stochastic stats ok: {n_st}")

# persist the per-galaxy CIGALE-model (delayed+bq) fits: Part 7d uses them as
# the SFH-parameter truth (same functional form CIGALE fits, so the comparison
# is apples-to-apples — the SFT/QT clocks are a different quantity).
SFH_FITS_TABLE = os.path.join(TABLEDIR, "sfh_delayedbq_fits.fits")
_ft = Table(dict(
    snap=[g["snap"] for g in SFH], gal_id=[g["gal_id"] for g in SFH],
    z=[g["z"] for g in SFH],
    tau_main_myr=[g["bq"]["tau_main_myr"] if g["bq"] else np.nan for g in SFH],
    age_main_myr=[g["bq"]["age_main_myr"] if g["bq"] else np.nan for g in SFH],
    age_bq_myr=[g["bq"]["age_bq_myr"] if g["bq"] else np.nan for g in SFH],
    r_sfr=[g["bq"]["r_sfr"] if g["bq"] else np.nan for g in SFH],
    r2=[g["bq"]["r2"] if g["bq"] else np.nan for g in SFH],
    sigma_dex=[g["st"]["sigma_dex"] if g["st"] else np.nan for g in SFH],
    tau_break_myr=[g["st"]["tau_break_myr"] if g["st"] else np.nan for g in SFH],
))
_ft.write(SFH_FITS_TABLE, overwrite=True)
print(f"[SFH priors] per-galaxy delayed+bq fits -> {SFH_FITS_TABLE}")


def _pc(vals, q=(5, 50, 95)):
    v = np.asarray(vals, float); v = v[np.isfinite(v)]
    return tuple(np.percentile(v, q)) if v.size else (np.nan,) * len(q)


_log_mstar = [g["log_mstar"] for g in SFH]
_log_ssfr = [np.log10(g["ssfr"]) for g in SFH if g["ssfr"] > 0]
_tau_q = [g["tau_q_gyr"] * 1e3 for g in SFH]
_age_bq_emp = [g["age_bq_emp_myr"] for g in SFH]
_fit_tau = [g["bq"]["tau_main_myr"] for g in SFH if g["bq"]]
_fit_agem = [g["bq"]["age_main_myr"] for g in SFH if g["bq"]]
_fit_agebq = [g["bq"]["age_bq_myr"] for g in SFH if g["bq"]]
_fit_rsfr = [g["bq"]["r_sfr"] for g in SFH if g["bq"]]
_fit_tau1 = [g["e2"]["tau1_myr"] for g in SFH if g["e2"]]
_fit_tau2 = [g["e2"]["tau2_myr"] for g in SFH if g["e2"]]
_st_sigma = [g["st"]["sigma_dex"] for g in SFH if g["st"]]
_st_taub = [g["st"]["tau_break_myr"] for g in SFH if g["st"]]
_st_dt = np.median([g["st"]["dt_myr"] for g in SFH if g["st"]]) if _st_sigma else np.nan

print("\n TRUE sample parameter ranges  (p5 / p50 / p95):")
def _line(name, vals, unit="", nd=2):
    a, b, c = _pc(vals)
    print(f"   {name:<26s} {a:9.{nd}f} {b:9.{nd}f} {c:9.{nd}f}  {unit}")
_line("log10 M*/Msun", _log_mstar)
_line("log10 sSFR/yr", _log_ssfr)
_line("tau_q (quench duration)", _tau_q, "Myr", 0)
_line("age_bq = t_obs - t_qt", _age_bq_emp, "Myr", 0)
print("   --- delayed-tau + burst/quench fits: ---")
_line("tau_main", _fit_tau, "Myr", 0)
_line("age_main", _fit_agem, "Myr", 0)
_line("age_bq", _fit_agebq, "Myr", 0)
_line("r_sfr", _fit_rsfr, "", 3)
print("   --- double-exponential fits: ---")
_line("tau1 (fast)", _fit_tau1, "Myr", 0)
_line("tau2 (slow)", _fit_tau2, "Myr", 0)
print("   --- stochastic SFH burst statistics (Carvajal+2025): ---")
_line("sigma (burst amp)", _st_sigma, "dex", 3)
_line("tau_break (decorr)", _st_taub, "Myr", 0)

_cur = DEFAULT_MODULE_PARAMS["sfhdelayedbq"]
print("\n suggested sfhdelayedbq grid (p5..p95 of the fits) vs current nodes:")
for par, vals in [("tau_main", _fit_tau), ("age_main", _fit_agem),
                  ("age_bq", _fit_agebq), ("r_sfr", _fit_rsfr)]:
    a, b, c = _pc(vals)
    print(f"   {par:<9s} data p5/50/95 = {a:8.1f} /{b:8.1f} /{c:8.1f}    "
          f"current nodes = {_cur.get(par)}")
_sa, _sb, _sc = _pc(_st_sigma); _ta, _tb, _tc = _pc(_st_taub)
print("\n suggested sfhstochastic_carvajal2025 grid (data p5/50/95):")
print(f"   sigma      = {_sa:.2f} /{_sb:.2f} /{_sc:.2f} dex   (SFR burst amplitude)")
print(f"   tau_break  = {_ta:.0f} /{_tb:.0f} /{_tc:.0f} Myr   "
      f"(NOTE: unresolved below the snapshot cadence ~{_st_dt:.0f} Myr)")
print(f"   alpha      = keep ~2 (damped random walk); a PSD slope is not "
      "reliably measurable from ~snapshot-cadence SFHs")
print(f"   tau_main / age = from the delayed fits above (baseline delayed SFH)")

# ── figure 1: example galaxy SFHs with both fits ──
_good = [g for g in SFH if g["bq"]]
_good = sorted(_good, key=lambda g: g["log_mstar"])
_pick = ([_good[int(x)] for x in np.linspace(0, len(_good) - 1, min(6, len(_good)))]
         if _good else [])
if _pick:
    _nc = min(3, len(_pick)); _nr = int(np.ceil(len(_pick) / _nc))
    fig1, ax1 = plt.subplots(_nr, _nc, figsize=(4.6 * _nc, 3.4 * _nr),
                             squeeze=False)
    for k, g in enumerate(_pick):
        ax = ax1[k // _nc][k % _nc]
        ax.plot(g["t_raw"], g["sfr_raw"], "o", ms=3.5, color="0.65",
                label="SIMBA snapshots")
        ax.plot(g["t"], g["sfr"], "-", color="0.25", lw=1.4,
                label="smoothed+resampled")
        tt = np.linspace(g["t"][0], g["t"][-1], 200)
        b = g["bq"]
        ax.plot(tt, sfr_delayed_bq(tt, b["A"], b["tau_main_myr"] / 1e3,
                b["age_main_myr"] / 1e3, b["age_bq_myr"] / 1e3, b["r_sfr"],
                b["t_obs"]), "-", color="#D55E00", lw=1.8,
                label="delayed+bq")
        if g["e2"]:
            e = g["e2"]
            ax.plot(tt, sfr_2exp(tt, e["A1"], e["tau1_myr"] / 1e3, e["A2"],
                    e["tau2_myr"] / 1e3, e["t0"]), "--", color="#0072B2",
                    lw=1.5, label="2-exp")
        if g["st"]:                                   # stochastic: trend + burst band
            s = g["st"]
            _ts = np.linspace(s["t0"], s["t1"], 100)
            _trend = 10 ** np.polyval(s["poly"], _ts)
            _fac = 10 ** s["sigma_dex"]
            ax.plot(_ts, _trend, "-", color="#009E73", lw=1.2, label="stoch. trend")
            ax.fill_between(_ts, _trend / _fac, _trend * _fac, color="#009E73",
                            alpha=0.18, label=r"$\pm\sigma$ burst")
        ax.axvline(g["t_obs"] - b["age_bq_myr"] / 1e3, color="0.6", ls=":", lw=1)
        _sig = g["st"]["sigma_dex"] if g["st"] else np.nan
        ax.set_title(f"id {g['gal_id']} z={g['z']:.2f}  "
                     r"$\tau$=%.0f age$_{bq}$=%.0f r=%.2f $\sigma$=%.2f"
                     % (b["tau_main_myr"], b["age_bq_myr"], b["r_sfr"], _sig),
                     fontsize=8)
        ax.set_xlabel("cosmic time [Gyr]"); ax.set_ylabel(r"SFR [$M_\odot$/yr]")
        if k == 0:
            ax.legend(fontsize=7, frameon=False)
    for k in range(len(_pick), _nr * _nc):
        ax1[k // _nc][k % _nc].set_axis_off()
    fig1.suptitle("example true SFHs with delayed+bq and double-exp fits", y=1.0)
    fig1.tight_layout()
    fig1.savefig(os.path.join(PLOTDIR, "sfh_prior_examples.png"),
                 dpi=150, bbox_inches="tight")

# ── figure 2: median stacked SFH + fitted-parameter distributions vs grid ──
_L = np.linspace(0, 8, 60)          # lookback from observation [Gyr]
_stack = []
for g in SFH:
    look = g["t_obs"] - g["t"]; pk = np.nanmax(g["sfr"])
    if pk <= 0:
        continue
    o = np.argsort(look)
    _stack.append(np.interp(_L, look[o], (g["sfr"] / pk)[o], left=np.nan,
                            right=np.nan))
_stack = np.array(_stack)
fig2, ax2 = plt.subplots(2, 4, figsize=(19, 8)); ax2 = ax2.ravel()
if _stack.size:
    med = np.nanmedian(_stack, 0)
    q16, q84 = np.nanpercentile(_stack, [16, 84], 0)
    ax2[0].fill_between(_L, q16, q84, color="#0072B2", alpha=0.25,
                        label="16-84%")
    ax2[0].plot(_L, med, "-", color="#0072B2", lw=2, label="median")
    _tobs_med = float(np.median([g["t_obs"] for g in SFH]))
    _fb = fit_delayed_bq(_tobs_med - _L[np.isfinite(med)][::-1],
                         med[np.isfinite(med)][::-1], _tobs_med)
    if _fb:
        _tt = _tobs_med - _L
        ax2[0].plot(_L, sfr_delayed_bq(_tt, _fb["A"], _fb["tau_main_myr"] / 1e3,
                    _fb["age_main_myr"] / 1e3, _fb["age_bq_myr"] / 1e3,
                    _fb["r_sfr"], _tobs_med), "-", color="#D55E00", lw=1.8,
                    label="delayed+bq fit")
    ax2[0].set_xlabel("lookback from observation [Gyr]")
    ax2[0].set_ylabel("SFR / peak")
    ax2[0].set_title("median stacked SFH"); ax2[0].legend(fontsize=7,
                                                          frameon=False)

# histograms: delayed+bq params (grid nodes overlaid) + stochastic params
for ax, par, vals, unit, nodes in [
        (ax2[1], "tau_main", _fit_tau, "Myr", _cur.get("tau_main")),
        (ax2[2], "age_main", _fit_agem, "Myr", _cur.get("age_main")),
        (ax2[3], "age_bq", _fit_agebq, "Myr", _cur.get("age_bq")),
        (ax2[4], "r_sfr", _fit_rsfr, "", _cur.get("r_sfr")),
        (ax2[5], "sigma (stochastic)", _st_sigma, "dex", None),
        (ax2[6], "tau_break (stochastic)", _st_taub, "Myr", None)]:
    v = np.asarray(vals, float); v = v[np.isfinite(v)]
    if v.size:
        ax.hist(v, bins=20, color="0.7", edgecolor="0.4")
    for node in (nodes or []):
        ax.axvline(node, color="#D55E00", ls="--", lw=1.2)
    if par.startswith("tau_break") and np.isfinite(_st_dt):
        ax.axvline(_st_dt, color="#0072B2", ls=":", lw=1.5)
        ax.text(0.97, 0.95, "snapshot\ncadence", transform=ax.transAxes,
                fontsize=7, ha="right", va="top", color="#0072B2")
    ax.set_xlabel(f"{par} [{unit}]" if unit else par)
    ax.set_ylabel("galaxies")
    _t = (f"{par}: fits vs grid nodes" if nodes else f"{par}: data (no grid yet)")
    ax.set_title(_t, fontsize=9)
ax2[7].set_axis_off()
fig2.suptitle("SFH-parameter distributions (bars): delayed+bq vs current grid "
              "nodes (dashed) + stochastic burst stats", y=1.0)
fig2.tight_layout()
fig2.savefig(os.path.join(PLOTDIR, "sfh_prior_distributions.png"),
             dpi=150, bbox_inches="tight")
print(f"\n[SFH priors] figures -> {PLOTDIR}/sfh_prior_examples.png, "
      "sfh_prior_distributions.png")
plt.show()

# Part 7c — configure & run CIGALE (**cluster**)

Fully replaces `pcigale init` → edit → `genconf` → edit → `check` → `run`:
`simbanator.sed.cigale.prepare_run` writes a complete, validated `pcigale.ini` + `.spec`
(bands auto-read from the data file, every parameter documented with its genconf-style
comment), and `run` executes the fit. One run directory per (dust mode, aperture) under
`output/cis25/cigale_runs/`; results in each `…/out/results.fits`.

**Runs on the cluster.** CIGALE 2025.0 lives in its **own** conda env — do NOT install it into
`pd39` (powderday pins numpy/astropy). One-time setup on a login node:

```bash
conda create -n cigale python=3.12 -y
conda activate cigale
pip install <path to the cigale-v2025.0 tarball from cigale.lam.fr>   # or `pip install .` in the unpacked dir
```

The notebook kernel stays `pd39`: only the env's `pcigale` executable is needed, and the cell
finds it by absolute path (no `conda activate` at runtime, so it also works in Slurm scripts).

**In-notebook reminder.** `cg.describe_run()` prints the full module/parameter/variable
documentation (what `pcigale genconf` would put as comments in the ini) before the loop; after
each `prepare_run` a compact `describe_run(run_dir, docs=False)` echoes the grids that will
actually run.

**Legacy-catalog repair.** Part 7b files written before the `convolveFilterWithSED` sign fix
(2026-07-07) carry **all-negative errors** (descending-λ hyperion SEDs made `trapz` norms
negative). CIGALE reads a negative error as "upper limit" → every band an upper limit → an
all-NaN `results.fits`. The preflight below `abs()`es such files in place and says so.

**Defaults** (all overridable via `module_params=` / `analysis_params=` in `prepare_run`):
`sfhdelayedbq` (delayed SFH + burst/quench episode — `age_bq`/`r_sfr` map directly onto the
quench window of this sample) + `bc03` (Chabrier, Z = 0.008/0.02/0.05) + `nebular` +
`dustatt_modified_CF00` (Av_ISM 0→2) + `dl2014` + `redshifting`; `additionalerror = 0.1`;
saved variables include `stellar.m_star`, `sfh.sfr*`, the full SFH shape
(`sfh.tau_main`/`age_main`/`age_bq`/`r_sfr`), `stellar.age_m_star` (mass-weighted age) and
`attenuation.Av_ISM` — the set Part 7d compares. **Runs fitted before 2026-07-09 lack the
age/A_V columns — re-run `prepare_run` + `run` to refresh.** That grid is **126 000 models per redshift** — fine for one anchor,
noticeable for five; trim the grids in `module_params` if runtime matters.

In [ ]:
# # ── Part 7c: prepare + run CIGALE on every input file (CLUSTER; kernel stays pd39) ──
# import shutil
# from simbanator.sed import cigale as cg

# # pcigale executable from the dedicated env (one-time setup: see the markdown above)
# _pc_cands = [os.path.expanduser(f"~/{_r}/envs/{_e}/bin/pcigale")
#              for _r in ("miniforge3", "miniconda3")
#              for _e in ("cigale", "cigale-env")] + [shutil.which("pcigale")]
# PCIGALE_CMD = next((p for p in _pc_cands if p and os.path.exists(p)), None)
# assert PCIGALE_CMD, (f"pcigale not found ({_pc_cands[:-1]} + PATH) — "
#                      "create the 'cigale' conda env first (markdown above)")
# print("pcigale:", PCIGALE_CMD)

# CIGALE_IN = os.path.join(CATDIR, "cigale")           # Part 7b catalogs
# RUN_BASE  = os.path.join(OUT, "cigale_runs")
# PLOT_SEDS = True    # one best-fit SED figure per object -> <run_dir>/out/<id>_best_model.png
# SKIP_IF_DONE = False  # False: re-fit, pcigale keeps the old out/ as <timestamp>_out/;
#                       # True: return existing results.fits untouched (cheap re-runs)

# # reminder: every module, parameter and analysis variable (genconf-style docs)
# cg.describe_run()

# CIGALE_RESULTS = {}
# for data_file in sorted(glob.glob(os.path.join(CIGALE_IN, "cigale_*.fits"))):
#     tag = os.path.basename(data_file)[len("cigale_"):-len(".fits")]   # e.g. dust_on_ap10kpc
#     if tag.startswith("fluxes_"):
#         continue                                     # Part 7b intermediates, not data files

#     # preflight: legacy catalogs (pre sign-fix) have all-negative errors -> CIGALE
#     # would treat every band as an upper limit and return an all-NaN results.fits
#     _t = Table.read(data_file)
#     _ecols = [c for c in _t.colnames if c.endswith("_err")]
#     _nneg = int(sum((np.asarray(_t[c], float) < 0).sum() for c in _ecols))
#     if _nneg:
#         for c in _ecols:
#             _t[c] = np.abs(np.asarray(_t[c], float))
#         _t.write(data_file, overwrite=True)
#         print(f"[repair] {tag}: {_nneg} negative error(s) -> abs(), file rewritten")

#     run_dir = os.path.join(RUN_BASE, tag)
#     print(f"\n=== {tag} ===")
#     cg.prepare_run(run_dir, data_file)   # defaults: sfhdelayedbq+bc03+nebular+CF00+dl2014
#     # to shrink the grid, e.g.:
#     # cg.prepare_run(run_dir, data_file, module_params={
#     #     "sfhdelayedbq": {"tau_main": [1000, 2000], "age_bq": [300, 1000]},
#     #     "dustatt_modified_CF00": {"Av_ISM": [0.0, 0.5, 1.0]}})
#     cg.describe_run(run_dir, docs=False)             # compact echo of the grids that will run
#     CIGALE_RESULTS[tag] = cg.run(run_dir, pcigale_cmd=PCIGALE_CMD, skip_if_done=SKIP_IF_DONE)
#     if PLOT_SEDS:
#         cg.plot_seds(run_dir, format="png",
#                      pcigale_plots_cmd=PCIGALE_CMD + "-plots")

# print("\nresults:")
# for tag, path in CIGALE_RESULTS.items():
#     print(f"  {tag}: {path}")

# Part 7c′ — CIGALE as a SLURM job array (parallel; recommended for the full 72)

The Part 7c loop fits the input files **sequentially in the notebook kernel**. This variant
prepares every run directory the same way (preflight repair + `prepare_run`) and then submits
**one SLURM job array with one task per run** (`simbanator.sed.cigale.write_slurm_array`).

Why per-run and not per-object: CIGALE builds its model grid (~126 000 models × n_z) **once
per run** and shares it across all objects in the data file — splitting a catalog by rows
would recompute the same grid in every chunk. The parallelism budget is therefore
72 array tasks × `CORES_PER_TASK` pcigale worker processes each; use `array_throttle` to
be polite on a busy cluster.

Each task rewrites `cores` in `pcigale.ini` to the actual `$SLURM_CPUS_PER_TASK`, sets
`OMP_NUM_THREADS=1`, runs the fit, then `pcigale-plots sed` (plot failures don't fail the
task). Logs land in `cigale_runs/slurm_logs/`. **Re-running is the default**
(`SKIP_IF_DONE = False`): a pre-existing `out/` is renamed by CIGALE itself to a timestamped
`<YYYYMMDDHHMM>_out/` backup, exactly like a manual `pcigale run` — nothing is overwritten.
Set `SKIP_IF_DONE = True` (and re-run this cell to regenerate the script) when resubmitting
after a partial failure, so finished runs exit immediately and only the missing ones fit.

**Metallicity priors (`USE_Z_PRIORS`).** With Part 4c's `aperture_metallicities.fits`
present, each catalog is split into per-metallicity sub-runs
(`simbanator.sed.cigale.split_by_metallicity`): every galaxy's SIMBA Z_star/Z_gas **in that
catalog's aperture/annulus and sightline** is snapped to the nearest allowed CIGALE grid value
(log-space; option lists parsed from the module registry = verbatim CIGALE 2025 sources).
Groups are keyed by the **bc03 node only** (6 coarse values → few groups): each is fitted with
that single stellar Z, while its nebular `zgas` grid is restricted to **at most 3** values
(quantiles of the group members' snapped zgas, re-snapped to the grid). Both caps matter: the
zgas grid has 26 dense nodes, so keying groups on it would fragment the sample into per-object
runs, and letting a group collect every member's zgas once ballooned a task to 84M models
(14 zgas × 10⁴ SFH × 120 dust × 5 z) → one shared-memory block → **SIGBUS** on the node.
`prepare_run` now also auto-sets CIGALE's `blocks` so no block exceeds ~5M models (~4 GB of
`/dev/shm`, which SLURM counts against the job's memory). Run dirs gain a `_Zs<z>` suffix; galaxies with no measurable Z (empty annulus) fall
into a `_Zsfree` sub-run on the default grid.

**Per-snapshot runs & the sfhdelayedbq constraint.** Every model must satisfy
`age_bq < age_main` (`t_bq = age_main − age_bq` indexes the SFR array: `age_bq > 2·age_main`
crashes with an IndexError, and `age_main ≤ age_bq ≤ 2·age_main` **silently wraps into a
garbage SFH** — no error, poisoned posteriors). A Cartesian ini cannot express that
constraint, so each Z-group is further split **by anchor snapshot** and the age grids are
built per redshift: `age_main` = fractions of t_universe(z) (`AGE_MAIN_FRACS`), `age_bq`
log-spaced up to `AGE_BQ_MAX_FRAC`·t_universe < min(age_main). This also keeps every model
younger than the universe at its own z. Run dirs: `<tag>_snap<NNN>_Zs<z>`.

Workflow: run this cell → `sbatch output/cis25/cigale_runs/submit_cigale_array.job` on a
login node → when the array drains, Part 7d works unchanged. Part 7c and 7c′ are
interchangeable (same run dirs, same skip logic) — use 7c for a quick single-tag test,
7c′ for the full set.

In [ ]:
# ── Part 7c′: prepare all runs + ONE SLURM job array (cluster; alternative to 7c) ──
import re
import shutil
from simbanator.sed import cigale as cg

# pcigale from the dedicated env (miniforge3 on this cluster; see Part 7c markdown)
_pc_cands = [os.path.expanduser(f"~/{_r}/envs/{_e}/bin/pcigale")
             for _r in ("miniforge3", "miniconda3") for _e in ("cigale", "cigale-env")]
_pc_cands += [shutil.which("pcigale")]
PCIGALE_CMD = next((p for p in _pc_cands if p and os.path.exists(p)), None)
assert PCIGALE_CMD, (f"pcigale not found ({_pc_cands[:-1]} + PATH) — "
                     "create the 'cigale' conda env first (Part 7c markdown)")
print("pcigale:", PCIGALE_CMD)

CIGALE_IN = os.path.join(CATDIR, "cigale")           # Part 7b + 7b'' input files
RUN_BASE  = os.path.join(OUT, "cigale_runs")
CORES_PER_TASK = 8                                   # pcigale workers per array task
PLOT_SEDS = True                                     # per-object best-fit SED figures
SKIP_IF_DONE = False                                 # False: re-fit, old out/ kept as
                                                     #   <timestamp>_out/ (CIGALE-native);
                                                     # True: finished runs exit at once
USE_Z_PRIORS = True                                  # per-galaxy Z from Part 4c -> Z sub-runs

# ── age grids are PER SNAPSHOT (runs are split by anchor): sfhdelayedbq requires
# age_bq < age_main in EVERY grid combination (t_bq = age_main - age_bq indexes the
# SFR array: age_bq > 2*age_main crashes, age_main <= age_bq <= 2*age_main silently
# wraps into a garbage SFH), and a Cartesian ini cannot express that constraint —
# so the grids are built with max(age_bq) < min(age_main) by construction, anchored
# to the universe age at that snapshot's z (also keeps every model younger than
# the universe). Tune the fractions here.
AGE_MAIN_FRACS  = np.linspace(0.55, 0.95, 5)   # SF onset: fractions of t_universe(z)
AGE_BQ_MAX_FRAC = 0.5                          # latest b/q lookback: < min(AGE_MAIN_FRACS)
N_AGE_BQ        = 10

def _age_grids(z):
    t_uni = float(COSMO.age(z).to_value("Myr"))
    age_main = np.unique((AGE_MAIN_FRACS * t_uni).round().astype(int))
    age_bq = np.unique(np.logspace(1, np.log10(AGE_BQ_MAX_FRAC * t_uni),
                                   N_AGE_BQ).round().astype(int))
    assert age_bq.max() < age_main.min()
    return age_main, age_bq

def _module_params(z):
    age_main, age_bq = _age_grids(z)
    return {
        "sfhdelayedbq": {"tau_main": np.logspace(2, 4.2, 10),
                         "age_main": age_main,
                         "age_bq": age_bq,
                         "r_sfr": np.linspace(0, 2.3, 10)},
        "dustatt_modified_CF00": {"Av_ISM": np.linspace(0, 4.2, 10)},
    }

# per-galaxy metallicity priors (Part 4c): Zstar -> bc03 grid, Zgas -> nebular zgas grid
_ztabf = os.path.join(TABLEDIR, "aperture_metallicities.fits")
_ztab = Table.read(_ztabf) if (USE_Z_PRIORS and os.path.exists(_ztabf)) else None
if USE_Z_PRIORS and _ztab is None:
    print(f"[Z priors] {_ztabf} missing — run Part 4c first; fitting WITHOUT Z priors")

def _z_maps(label, ilab):
    # {id: Z} for stars and gas in this catalog's aperture/annulus + sightline
    _m = np.char.strip(np.asarray(_ztab["incl"], str)) == ilab
    _ids = [f"snap{int(s):03d}_gal{int(g)}" for s, g in
            zip(np.asarray(_ztab["snap"], int)[_m], np.asarray(_ztab["gal_id"], int)[_m])]
    _zs = np.asarray(_ztab[f"Zstar_{label}"], float)[_m]
    _zg = np.asarray(_ztab[f"Zgas_{label}"], float)[_m]
    return dict(zip(_ids, _zs)), dict(zip(_ids, _zg))

PERSNAP_DIR = os.path.join(CIGALE_IN, "persnap")
os.makedirs(PERSNAP_DIR, exist_ok=True)

_run_dirs = []
for data_file in sorted(glob.glob(os.path.join(CIGALE_IN, "cigale_*.fits"))):
    tag = os.path.basename(data_file)[len("cigale_"):-len(".fits")]
    if tag.startswith("fluxes_"):
        continue                                     # Part 7b intermediates, not data files

    # same preflight as Part 7c: legacy all-negative errors -> all-NaN fits
    _t = Table.read(data_file)
    _ecols = [c for c in _t.colnames if c.endswith("_err")]
    _nneg = int(sum((np.asarray(_t[c], float) < 0).sum() for c in _ecols))
    if _nneg:
        for c in _ecols:
            _t[c] = np.abs(np.asarray(_t[c], float))
        _t.write(data_file, overwrite=True)
        print(f"[repair] {tag}: {_nneg} negative error(s) -> abs(), file rewritten")

    _mt = re.match(r"^(dust_on|dust_off)_((?:ap|ann)[0-9]+kpc)_(i\d+p\d+)$", tag)
    _zs_map = _zg_map = None
    if _ztab is not None and _mt is not None:
        _zs_map, _zg_map = _z_maps(_mt.group(2), _mt.group(3))
    elif _ztab is not None:
        print(f"[Z priors] {tag}: tag not (ap|ann)*kpc_i*p* — default Z grid")

    # one run per (snapshot, metallicity group): single z per run -> the per-z age
    # grids above apply to every object of the run
    _tsnaps = np.array([int(str(i).split("_")[0][4:]) for i in _t["id"]])
    for _sn in sorted(set(_tsnaps.tolist())):
        _mrows = _tsnaps == _sn
        _sntag = f"{tag}_snap{_sn:03d}"
        _snapf = os.path.join(PERSNAP_DIR, f"cigale_{_sntag}.fits")
        _t[_mrows].write(_snapf, overwrite=True)
        _z = float(np.asarray(_t["redshift"], float)[_mrows][0])
        _base = _module_params(_z)

        if _zs_map is not None:
            groups = cg.split_by_metallicity(_snapf, _zs_map, zgas=_zg_map)
        else:
            groups = [{"tag": "", "path": _snapf, "module_params": None}]

        for grp in groups:
            run_dir = os.path.join(RUN_BASE,
                                   _sntag + (f"_{grp['tag']}" if grp["tag"] else ""))
            _mp = dict(_base)
            _mp.update(grp["module_params"] or {})   # bc03/nebular keys are disjoint
            cg.prepare_run(run_dir, grp["path"], cores=CORES_PER_TASK,
                           module_params=_mp, verbose=False)
            _run_dirs.append(run_dir)

print(f"{len(_run_dirs)} run dirs prepared under {RUN_BASE}")
JOB_FILE = cg.write_slurm_array(
    _run_dirs, os.path.join(RUN_BASE, "submit_cigale_array.job"),
    pcigale_cmd=PCIGALE_CMD,
    partition='INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL',
    cores=CORES_PER_TASK, time="0-12:00",
    array_throttle=None,                # e.g. 32 to cap simultaneous tasks
    plots=PLOT_SEDS, skip_if_done=SKIP_IF_DONE)
# -> sbatch it on a login node. Re-running the cell regenerates the script, so flip
#    SKIP_IF_DONE=True before re-sbatching if you only want the missing runs.

# Part 7d·prep — aperture-matched SIMBA truth (cluster, cached once)

CIGALE's estimates describe the stars inside **one projected aperture along one sightline** —
the global caesar/history M\*/SFR/age are only the right truth for the largest aperture. One
pass over the Stage-0 region cutouts measures, per (galaxy, sightline, aperture rung):

- **M\*** — current stellar mass in the projected cylinder (matching Hyperion's peeled
  apertures: radius r perpendicular to the (θ, φ) sightline, full depth);
- **archaeological SFR** over the last 25 / 100 Myr — mass formed in the window from
  `StellarFormationTime` (current masses, so ≲10–15 % mass-loss bias — noted, not corrected);
- **mass-weighted age and total Z** of the same stars (→ `stellar.age_m_star`,
  `stellar.metallicity`);
- a **delayed+bq fit to the aperture's own archaeological SFH** (50 Myr bins,
  `simbanator.analysis.sfh_utils.fit_delayed_bq` — the same form CIGALE fits), giving the
  per-aperture `sfh.tau_main / age_main / age_bq / r_sfr` truth.

Everything is cached to `tables/aperture_truth.fits` (one row per galaxy × sightline ×
aperture; `OVERWRITE_APERTURE_TRUTH=True` rebuilds). Part 7d overlays this cache on its truth
table per run; the global values remain the fallback when the cache is absent. Requires the
Stage-0 particle files and the Stage-1 selection HDF5 (centres = the RT-grid `x_cent`), so it
runs on the **cluster**.


In [ ]:
# ── Part 7d·prep — cache SIMBA properties in the SAME apertures/sightlines as the RT ──
from simbanator.analysis.sfh_utils import fit_delayed_bq

APERTURE_TRUTH_FITS      = os.path.join(TABLEDIR, "aperture_truth.fits")
OVERWRITE_APERTURE_TRUTH = False
SFR_WINDOWS_MYR = (25.0, 100.0)     # -> sfh.sfr / sfh.sfr100Myrs truth
ARCH_BIN_MYR    = 50.0              # archaeological-SFH bin for the delayed+bq fit
NSTAR_AP_MIN    = 20                # ages/Z/fits need at least this many star particles

if os.path.exists(APERTURE_TRUTH_FITS) and not OVERWRITE_APERTURE_TRUTH:
    APERTURE_TRUTH = Table.read(APERTURE_TRUTH_FITS)
    print(f"cached ({len(APERTURE_TRUTH)} rows) -> {APERTURE_TRUTH_FITS}  "
          "(OVERWRITE_APERTURE_TRUTH=True rebuilds)")
else:
    SEL = Table.read(SELECTION_FITS)
    # RT-grid centres (code units) from the Stage-1 selection HDF5
    _selh5 = os.path.join(sed_output_dir, RUNS["dust_on"]["run_tag"],
                          "target_selection", selection_file + ".h5")
    _cen = {}
    with h5py.File(_selh5, "r") as f:
        for _grp in f:
            _sn = int(_grp[4:])
            for _g, _p in zip(f[f"{_grp}/galaxy_GroupID"][:], f[f"{_grp}/code_coods"][:]):
                _cen[(_sn, int(_g))] = np.asarray(_p, float)

    # sightline unit vectors (Hyperion peel directions) + a->t interpolation grid
    _th, _ph = np.deg2rad(THETA_DEG), np.deg2rad(PHI_DEG)
    _nvec = np.stack([np.sin(_th) * np.cos(_ph), np.sin(_th) * np.sin(_ph),
                      np.cos(_th)], axis=1)
    _ag = np.linspace(0.02, 1.0, 4096)
    _tg = COSMO.age(1.0 / _ag - 1.0).value                      # Gyr

    _rows = []
    _apr = APERTURE_RADII_KPC[np.asarray(WANTED_AP_IDX)]        # true rung radii [pkpc]
    _snaps_sel = np.asarray(SEL["snap"], int)
    for _snap in np.unique(_snaps_sel):
        _zs = float(sim.get_z_from_snap(int(_snap))); _a = 1.0 / (1.0 + _zs)
        _t_obs = float(COSMO.age(_zs).value)
        _hdir = os.path.join(hydro_dir_base, f"snap_{_snap:03d}")
        _n_gal = 0
        for _gid in np.unique(np.asarray(SEL["gal_id"], int)[_snaps_sel == _snap]):
            _pf = os.path.join(_hdir, f"{PARTICLE_PREFIX}_snap{_snap:03d}_gal{_gid:06d}.h5")
            _c = _cen.get((int(_snap), int(_gid)))
            if _c is None or not os.path.exists(_pf):
                print(f"  [skip] snap {_snap} gal {_gid}: "
                      f"{'no centre in selection h5' if _c is None else 'no particle file'}")
                continue
            with h5py.File(_pf, "r") as f:
                if "PartType4" not in f:
                    print(f"  [skip] snap {_snap} gal {_gid}: no stars in cutout"); continue
                _h   = float(f["Header"].attrs["HubbleParam"])
                _pos = (np.asarray(f["PartType4/Coordinates"][:], float) - _c) * _a / _h
                _mst = np.asarray(f["PartType4/Masses"][:], float) * 1e10 / _h
                _af  = np.asarray(f["PartType4/StellarFormationTime"][:], float)
                _Zst = np.asarray(f["PartType4/Metallicity"][:], float)
                _Zst = _Zst[:, 0] if _Zst.ndim == 2 else _Zst
            _tf = np.interp(np.clip(_af, _ag[0], 1.0), _ag, _tg)    # formation time [Gyr]
            _n_gal += 1
            for _il, _nv in zip(INCL_LABELS, _nvec):
                _para = _pos @ _nv
                _rproj = np.sqrt(np.clip((_pos ** 2).sum(1) - _para ** 2, 0.0, None))
                for _lab, _r in zip(APERTURE_LABELS, _apr):
                    _msk = _rproj <= _r
                    _nap = int(_msk.sum())
                    _row = dict(snap=int(_snap), gal_id=int(_gid), incl=_il,
                                aperture=_lab, ap_kpc=float(_r), nstar_ap=_nap,
                                mstar=np.nan, sfr25=np.nan, sfr100=np.nan,
                                age_m_star_myr=np.nan, met_star=np.nan,
                                tau_main_myr=np.nan, age_main_myr=np.nan,
                                age_bq_myr=np.nan, r_sfr=np.nan, fit_r2=np.nan)
                    if _nap:
                        _mm, _tt, _zz = _mst[_msk], _tf[_msk], _Zst[_msk]
                        _row["mstar"] = float(_mm.sum())
                        for _w, _key in zip(SFR_WINDOWS_MYR, ("sfr25", "sfr100")):
                            _row[_key] = float(_mm[_tt >= _t_obs - _w / 1e3].sum()
                                               / (_w * 1e6))
                        if _nap >= NSTAR_AP_MIN:
                            _row["age_m_star_myr"] = float(
                                np.sum(_mm * (_t_obs - _tt)) / _mm.sum() * 1e3)
                            _row["met_star"] = float(np.sum(_mm * _zz) / _mm.sum())
                            _bins = np.arange(_tt.min(), _t_obs + ARCH_BIN_MYR / 1e3,
                                              ARCH_BIN_MYR / 1e3)
                            if _bins.size >= 8:
                                _hm, _ = np.histogram(_tt, bins=_bins, weights=_mm)
                                _tc = 0.5 * (_bins[1:] + _bins[:-1])
                                _fit = fit_delayed_bq(_tc, _hm / (ARCH_BIN_MYR * 1e6),
                                                      _t_obs)
                                if _fit:
                                    for _k2 in ("tau_main_myr", "age_main_myr",
                                                "age_bq_myr", "r_sfr"):
                                        _row[_k2] = float(_fit[_k2])
                                    _row["fit_r2"] = float(_fit["r2"])
                    _rows.append(_row)
        print(f"snap {_snap:3d} (z={_zs:.2f}): aperture truth for {_n_gal} galaxies")
    APERTURE_TRUTH = Table(rows=_rows)
    APERTURE_TRUTH.write(APERTURE_TRUTH_FITS, overwrite=True)
    print(f"{len(APERTURE_TRUTH)} rows ({len(INCL_LABELS)} sightlines x "
          f"{len(APERTURE_LABELS)} apertures) -> {APERTURE_TRUTH_FITS}")


# Part 7d — SIMBA truth vs CIGALE-recovered properties (per run)

For every finished run, `simbanator.sed.cigale.compare_results` joins `out/results.fits` with
the Part 3 selection catalog, **prints the per-galaxy table** (true vs recovered, log where
appropriate) with median-offset/NMAD stats, draws one-to-one panels (colored by anchor
redshift), and **saves both into that run's output folder**:
`<run_dir>/out/simba_vs_cigale.fits` + `.png`.

Compared properties (truth columns are named exactly like the CIGALE variables — add a column
here and it is compared automatically as long as the variable is in `prepare_run`'s
`variables`):

**Aperture-matched truth.** When `tables/aperture_truth.fits` exists (Part 7d·prep), every
truth property below except $A_V$ is replaced, per run, by the SIMBA value measured **inside
that run's projected aperture along that run's sightline** (M\*, archaeological SFR,
mass-weighted age & Z, delayed+bq fit of the aperture's own archaeological SFH). The global
caesar/history values below are only the fallback — they match only the largest aperture.

**Essential:**

- **`stellar.m_star`** — SIMBA stellar mass at the anchor vs CIGALE's Bayesian estimate.
- **`sfh.sfr`** / **`sfh.sfr100Myrs`** — SIMBA SFR from the **smoothed + resampled** SFH
  (same kernel treatment as Part 7b′, `simbanator.analysis.sfh_utils.recent_sfr`), averaged
  over the last 25 / 100 Myr before the anchor — the raw per-snapshot SFR is instantaneous
  and too stochastic to compare with an SED-derived estimate. For fully quenched fits
  CIGALE's posterior SFR underflows to ~0 — those points pin as open symbols at the plot
  floor.
- **`stellar.age_m_star`** — mass-weighted stellar age: caesar `ages.mass_weighted` (Gyr→Myr)
  vs CIGALE bc03's mass-weighted SSP age. Same definition on both sides.
- **`attenuation.Av_ISM`** — per run: the TRUE rest-frame Johnson-V attenuation
  $A_V = -2.5\log_{10}(F_{\rm on}/F_{\rm off})$ of that run's (aperture, sightline) catalog
  pair vs CF00's `Av_ISM` (for these old-star-dominated systems the effective V attenuation
  ≈ `Av_ISM`; young stars add `Av_BC` on top). dust_off runs use truth $A_V = 0$ — a null
  test that CIGALE recovers no attenuation from dust-free photometry.

**SFH shape** (`sfh.tau_main`, `sfh.age_main`, `sfh.age_bq`, `sfh.r_sfr`) — the truth is
**not** the SFT/QT clocks: it is the best fit of CIGALE's *own* `sfhdelayedbq` form to the
smoothed SIMBA SFH (Part 7b′ → `tables/sfh_delayedbq_fits.fits`), so both sides describe the
same model and the comparison isolates what the *SED* constrains, not the model mismatch.
Requires Part 7b′ to have run; its fit quality is carried as `sfh_fit_r2`, and the SFT/QT
clocks remain as context columns `age_sft_myr` / `age_qt_myr` (`t_sft`/`t_qt` are cosmic
times in **yr**).

**Dusty vs non-dusty.** The Part 7a′ dusty flag (global $A_V > 0.1$, same as Part 4b) is
joined into the truth table when `attenuation_vs_ism.fits` exists: dusty galaxies get a
**red ring** in every one-to-one panel (on top of the redshift colors) and the median-offset /
NMAD stats are printed per subsample — recovery biases that differ between the two are the
signature of attenuation-driven systematics in the fits. The `dusty` column is carried into
each saved `simba_vs_cigale.fits` (pass `highlight_col=None` to `compare_results` to disable).

Reading the output: `<prop>_cigale` is the Bayesian (posterior-mean) estimate with
`<prop>_cigale_err`; `<prop>_best` is the best-χ² model. **NaN bayes with finite best** means
the posterior weights underflowed because even the best model fits poorly — widen the grids in
`module_params` for those objects rather than trusting the best-fit value.

In [ ]:
# ── Part 7d: original SIMBA properties vs the CIGALE-recovered ones, per run ──
# Self-contained after Part 0: reads SELECTION_FITS, the anchor-history HDF5s in
# SFHDIR (smoothed truth SFR + mass-weighted age), the Part 7b' delayed+bq fit
# table (SFH-parameter truth) + each <run_dir>/out/results.fits.
import re
from simbanator.sed import cigale as cg

SEL = Table.read(SELECTION_FITS)
truth = Table()
truth["id"] = [f"snap{int(s):03d}_gal{int(g)}"
               for s, g in zip(SEL["snap"], SEL["gal_id"])]
truth["z_snap"]    = np.asarray(SEL["z_snap"], float)   # colors the one-to-one panels
truth["agn_class"] = SEL["agn_class"]                   # carried into the saved table
# truth columns named exactly like the CIGALE variables they are compared to:
truth["stellar.m_star"] = 10.0 ** np.asarray(SEL["log_mstar"], float)         # Msun
# truth SFR from the smoothed+resampled SFH (Part 7b' treatment) — the raw per-snapshot
# SFR is instantaneous and too stochastic to compare with an SED-derived estimate.
# 100 Myr average -> CIGALE sfh.sfr100Myrs; 25 Myr -> ~instantaneous sfh.sfr, de-burst-ed.
from simbanator.analysis.sfh_utils import recent_sfr
_sfhdb = {}
for _hf in sorted(glob.glob(os.path.join(SFHDIR, "history_anchor_*.hdf5"))):
    with h5py.File(_hf, "r") as f:
        _snap0 = int(f["metadata/snapshots"][0])               # row 0 = anchor epoch
        _gid   = np.asarray(f["metadata/galaxy_ids"][:], int)
        _tgyr  = COSMO.age(f["redshift/Redshift"][:]).value    # Gyr
        _sfr   = np.asarray(f["properties/sfr"][:], float)     # (n_snap, n_gal)
    _o = np.argsort(_tgyr)
    for _j, _g in enumerate(_gid):
        _sfhdb[(_snap0, int(_g))] = (_tgyr[_o], _sfr[_o, _j])

def _truth_sfr(avg_myr):
    out = np.full(len(SEL), np.nan)
    for _k, (_s, _g) in enumerate(zip(SEL["snap"], SEL["gal_id"])):
        _tr = _sfhdb.get((int(_s), int(_g)))
        if _tr is not None:
            out[_k] = recent_sfr(_tr[0], _tr[1], avg_myr=avg_myr)
    return out

truth["sfh.sfr"]        = _truth_sfr(25.0)
truth["sfh.sfr100Myrs"] = _truth_sfr(100.0)
_nsfr = int(np.isfinite(np.asarray(truth["sfh.sfr"], float)).sum())
print(f"truth SFR from smoothed SFHs: {_nsfr}/{len(SEL)} matched to anchor histories")
# mass-weighted stellar age at the anchor (caesar 'ages.mass_weighted', Gyr):
# the same definition as CIGALE's bc03 'stellar.age_m_star' (Myr)
_agedb = {}
for _hf in sorted(glob.glob(os.path.join(SFHDIR, "history_anchor_*.hdf5"))):
    with h5py.File(_hf, "r") as f:
        if "properties/ages.mass_weighted" not in f:
            continue
        _snap0 = int(f["metadata/snapshots"][0])
        _gid   = np.asarray(f["metadata/galaxy_ids"][:], int)
        _a0    = np.asarray(f["properties/ages.mass_weighted"][0], float)
    for _j, _g in enumerate(_gid):
        _agedb[(_snap0, int(_g))] = _a0[_j]
truth["stellar.age_m_star"] = np.array(
    [_agedb.get((int(s), int(g)), np.nan) * 1e3                       # Gyr -> Myr
     for s, g in zip(SEL["snap"], SEL["gal_id"])])

# SFH-parameter truth = the best fit of CIGALE's OWN sfhdelayedbq form to the
# smoothed SIMBA SFH (Part 7b' -> sfh_delayedbq_fits.fits). The SFT/QT clocks
# measure a different thing and are only carried as context columns.
_t_obs_yr = COSMO.age(truth["z_snap"]).value * 1e9      # t_sft / t_qt are in yr
truth["age_sft_myr"] = (_t_obs_yr - np.asarray(SEL["t_sft"], float)) / 1e6   # carried
truth["age_qt_myr"]  = (_t_obs_yr - np.asarray(SEL["t_qt"], float)) / 1e6    # carried
_fitf = os.path.join(TABLEDIR, "sfh_delayedbq_fits.fits")
if os.path.exists(_fitf):
    _ftab = Table.read(_fitf)
    _fdb = {(int(r["snap"]), int(r["gal_id"])): r for r in _ftab}
    for _tcol, _fcol in [("sfh.tau_main", "tau_main_myr"),
                         ("sfh.age_main", "age_main_myr"),
                         ("sfh.age_bq", "age_bq_myr"), ("sfh.r_sfr", "r_sfr")]:
        truth[_tcol] = np.array(
            [float(_fdb[(int(s), int(g))][_fcol]) if (int(s), int(g)) in _fdb
             else np.nan for s, g in zip(SEL["snap"], SEL["gal_id"])])
    truth["sfh_fit_r2"] = np.array(
        [float(_fdb[(int(s), int(g))]["r2"]) if (int(s), int(g)) in _fdb
         else np.nan for s, g in zip(SEL["snap"], SEL["gal_id"])])           # carried
    _nfit = int(np.isfinite(np.asarray(truth["sfh.age_bq"], float)).sum())
    print(f"SFH-parameter truth from {os.path.basename(_fitf)}: {_nfit}/{len(SEL)} fitted")
else:
    print(f"WARNING: {_fitf} missing — run Part 7b' first; "
          "sfh.tau_main/age_main/age_bq/r_sfr will not be compared")

# dusty split (Part 7a', global A_V > 0.1 at the fiducial aperture/sightline — the same
# flag as Part 4b): compare_results rings the dusty galaxies in every one-to-one panel and
# prints the offset/NMAD stats per subsample. -1 = unmatched (never ringed).
AV_DUSTY = 0.1
_avf = os.path.join(TABLEDIR, "attenuation_vs_ism.fits")
if os.path.exists(_avf):
    _at = Table.read(_avf)
    _avdb = {(int(s), int(g)): float(a) for s, g, a in
             zip(_at["snap"], _at["gal_id"], _at["A_V"])}
    _davg = np.array([_avdb.get((int(s), int(g)), np.nan)
                      for s, g in zip(SEL["snap"], SEL["gal_id"])])
    truth["dusty"] = np.where(np.isnan(_davg), -1, (_davg > AV_DUSTY).astype(int))
    print(f"dusty split (Part 7a' A_V > {AV_DUSTY:g}): {int((truth['dusty'] == 1).sum())} dusty"
          f" / {int((truth['dusty'] == 0).sum())} non-dusty"
          f" / {int((truth['dusty'] == -1).sum())} unmatched")
else:
    print(f"[dusty split] {_avf} missing — run Part 7a' first to ring the dusty galaxies")

# aperture-matched truth cache (Part 7d·prep): per (sightline, aperture) SIMBA
# properties measured exactly as CIGALE sees them; the global values above stay
# as the fallback when the cache is missing.
_apt_f = os.path.join(TABLEDIR, "aperture_truth.fits")
_APT = {}
if os.path.exists(_apt_f):
    for _r in Table.read(_apt_f):
        _APT[(str(_r["aperture"]), str(_r["incl"]),
              int(_r["snap"]), int(_r["gal_id"]))] = _r
    print(f"aperture-matched truth cache: {len(_APT)} rows from {os.path.basename(_apt_f)}")
else:
    print(f"WARNING: {_apt_f} missing — run Part 7d·prep; using GLOBAL truth for all runs")

RUN_BASE = os.path.join(OUT, "cigale_runs")
COMPARISONS = {}
for run_dir in sorted(glob.glob(os.path.join(RUN_BASE, "*"))):
    if not os.path.exists(os.path.join(run_dir, "out", "results.fits")):
        continue
    tag = os.path.basename(run_dir)
    if "_ann" in tag:
        continue   # annular runs: the global truth (total M*, SFR) is not per-annulus
    # per-run A_V truth: rest-frame Johnson-V attenuation from the matched
    # dust_on/dust_off catalogs of this run's (aperture, sightline); compared
    # to CF00's Av_ISM — for these old-star-dominated systems the effective
    # V attenuation ~ Av_ISM (young stars add Av_BC on top). dust_off runs
    # get truth A_V = 0 (null test: CIGALE should recover no attenuation).
    _truth_run = truth.copy()
    _m = re.match(r"^(dust_on|dust_off)_(.+)_(i\d+p\d+)(?:_snap\d+)?(?:_Zs.+)?$", tag)
    if _m is not None:
        _key, _lab, _il = _m.groups()
        _fon  = os.path.join(CATDIR, f"catalog_dust_on_{_lab}_{_il}.fits")
        _foff = os.path.join(CATDIR, f"catalog_dust_off_{_lab}_{_il}.fits")
        if os.path.exists(_fon) and os.path.exists(_foff):
            _on, _off = Table.read(_fon), Table.read(_foff)
            if "Johnson.V.V" in _on.colnames and "Johnson.V.V" in _off.colnames:
                _don = {(int(s), int(g)): float(v) for s, g, v in
                        zip(_on["snap"], _on["gal_id"], _on["Johnson.V.V"])}
                _dof = {(int(s), int(g)): float(v) for s, g, v in
                        zip(_off["snap"], _off["gal_id"], _off["Johnson.V.V"])}
                _av = np.full(len(SEL), np.nan)
                for _k, (_s, _g) in enumerate(zip(SEL["snap"], SEL["gal_id"])):
                    _fo = _don.get((int(_s), int(_g)))
                    _fx = _dof.get((int(_s), int(_g)))
                    if _fo and _fx and _fo > 0 and _fx > 0:
                        _av[_k] = -2.5 * np.log10(_fo / _fx)
                _truth_run["attenuation.Av_ISM"] = (
                    _av if _key == "dust_on" else np.zeros(len(SEL)))
    if _m is not None and _APT:
        # overlay the aperture-matched truth for this run's (aperture, sightline)
        _lab, _il = _m.group(2), _m.group(3)
        _cols = {"stellar.m_star": "mstar", "sfh.sfr": "sfr25",
                 "sfh.sfr100Myrs": "sfr100", "stellar.age_m_star": "age_m_star_myr",
                 "stellar.metallicity": "met_star",
                 "sfh.tau_main": "tau_main_myr", "sfh.age_main": "age_main_myr",
                 "sfh.age_bq": "age_bq_myr", "sfh.r_sfr": "r_sfr"}
        _nhit = 0
        for _tcol, _acol in _cols.items():
            _v = np.full(len(SEL), np.nan)
            for _k, (_s, _g) in enumerate(zip(SEL["snap"], SEL["gal_id"])):
                _r = _APT.get((_lab, _il, int(_s), int(_g)))
                if _r is not None:
                    _v[_k] = float(_r[_acol])
            if np.isfinite(_v).any():
                _truth_run[_tcol] = _v
                _nhit = max(_nhit, int(np.isfinite(_v).sum()))
        print(f"[{tag}] aperture-matched truth ({_lab}/{_il}): "
              f"{_nhit}/{len(SEL)} galaxies from the cache")
    elif not _APT:
        print(f"[{tag}] GLOBAL truth (no aperture cache)")
    print(f"\n{'=' * 25} {tag}: SIMBA vs CIGALE {'=' * 25}")
    COMPARISONS[tag] = cg.compare_results(run_dir, _truth_run)
    # -> prints per-galaxy table + offset/NMAD stats;
    #    saves <run_dir>/out/simba_vs_cigale.fits + .png

print(f"\n{len(COMPARISONS)} run(s) compared "
      f"-> simba_vs_cigale.fits/.png in each <run_dir>/out/")

# Run order (cheat sheet)

1. **cluster** — `BUILD_MULTI_Z=True` → run Parts 0–1 (histories); then `BUILD_BH=True` → Part 1
   BH cell. Flip both back to `False` afterwards.
2. Parts 2–3 (selection, AGN split, statistics, `powderday_quenched_selection.fits`) — needs only
   the HDF5s from step 1. Then **Part 3b** (mass–size QC): flags too-large / unresolved sources
   into `SELECTION_FITS` (`flag_too_large`, `flag_unresolved`; carried into every Part 7 catalog).
3. **cluster** — Part 4 (Stage 0 particle files), apply the **powderday aperture patch** (Part 5
   markdown), Part 5 cell, then `bash submit_all_snaps.sh` in **both** run trees under
   `output/cis25/sed_quenched_regions/<run_tag>/powderday_sed_out/`.
   Optional **Part 4b** (any time after Stage 0): star/gas/dust counts per projected annulus
   × sightline → `tables/annulus_particle_counts.fits` — the sampling QC behind the annular SEDs.
   **Part 4c**: mass-weighted Z_star/Z_gas per aperture & annulus × sightline →
   `tables/aperture_metallicities.fits` — the CIGALE metallicity priors (Part 7c′ Z sub-runs).
4. When `.rtout.sed` files exist: Part 6 QC (must show 5 apertures × 4 inclinations + MC
   uncertainties), then
   Part 7 → the per-aperture catalogs. Optional **Part 7a′**: differential dust attenuation
   ($A_V=-2.5\log_{10}F_{\rm on}/F_{\rm off}$) vs ISM ($f_{\rm mol}$, $M_{H_2}/M_\star$, dust/gas)
   and quench/AGN diagnostics — no CIGALE needed (`tables/attenuation_vs_ism.fits`). Then
   Part 7b → the CIGALE 2025.0 input files (`sed_aperture_catalogs/cigale/`). Then **Part 7b″** → the
   **annular** CIGALE inputs ($F(<r_{\rm out})-F(<r_{\rm in})$; labels `ann3kpc…ann100kpc`,
   `ann1kpc`≡`ap1kpc` not duplicated; star-free annuli flagged from Part 4b). Optional
   **Part 7b′**: derive the SFH priors from the true histories (parameter ranges +
   delayed-bq/2-exp/stochastic fits) to tune the Part 7c grid.
5. **cluster** — Part 7c: `prepare_run` + `run` fit every input file with CIGALE (its own
   `cigale` conda env, executable auto-found; kernel stays `pd39`); results in
   `output/cis25/cigale_runs/<tag>/out/results.fits` — or **Part 7c′** (recommended for the
   full set): prepare all run dirs, then ONE SLURM job array, one task per run
   (`sbatch cigale_runs/submit_cigale_array.job`; by default existing `out/` dirs are
   timestamped-and-kept by CIGALE and everything re-fits — `SKIP_IF_DONE=True` instead makes
   finished runs exit at once, for cheap resubmits after partial failures).
6. **cluster** — Part 7d·prep: one pass over the Stage-0 cutouts → `tables/aperture_truth.fits`
   (per galaxy × sightline × aperture: M*, archaeological SFR, mass-weighted age/Z, delayed+bq
   fit of the aperture SFH). Then Part 7d — SIMBA truth vs CIGALE estimates, **aperture-matched
   per run** (global truth is the fallback): per-galaxy print + one-to-one figures, saved as
   `simba_vs_cigale.fits/.png` into each `<run_dir>/out/`.

**Caveats.**
- Stage 0 writes **100 pkpc region cutouts** (CGM + satellites, periodic-wrap safe) under the
  same filenames as the old plist files (`EXTRACT_OVERWRITE=True` replaces them); the RT grid
  is ±100 kpc (`zoom_box_len`), and the 5 hyperion-log-spaced apertures (1, 3.16, 10, 31.6,
  100 kpc) sample central → outskirts. Only the outermost aperture is slightly depth-truncated
  at its edge (sphere inscribed in the cube). `N_AP/AP_MIN_KPC/AP_MAX_KPC` + `THETA_DEG/PHI_DEG`
  here must match `SED_APERTURE_*` / `THETA/PHI` in `simbanator/sed/parameters_master*.py` at
  RT time. The 1 kpc aperture spans only ~3–4 softening lengths (m25n512) — indicative only.
  Part 7c sees 2 dust modes × 5 apertures × 4 sightlines = 40 CIGALE inputs, plus
  2 × 4 annuli × 4 sightlines = 32 annular ones from Part 7b″ (72 total); filter the
  Part 7c glob (e.g. only `*_i0p0`) if runtime matters.
- `<filter>_err` is the Hyperion **Monte-Carlo photon noise** propagated through the filter
  convolution — it is an RT-convergence error, not a mock observational depth. All-NaN error
  columns mean the run stored no uncertainties (patch not applied / `set_uncertainties` missing).
- Fluxes are rest-frame convolved (`redshift=False`, as in `test_powderday.ipynb`); pass
  `redshift=True` in Part 7 for observed-frame photometry (Part 7b does this for CIGALE).
  Part 7a′'s $A_V$ is therefore a **rest-frame** attenuation, directly comparable across anchors.
- The dust_off run uses 1 dust-RT photon (`parameters_master-nodust.py`) — some galaxies can
  crash/truncate; the cross-check + `missing_sources_*.txt` make any loss explicit.
- CIGALE error budget: the input files carry raw MC errors; the fit adds `additionalerror = 0.1`
  (10 %) in quadrature via `prepare_run` — change it there, not in Part 7b.
- **Negative errors** = "upper limit" to CIGALE. Part 7b catalogs written before the
  `convolveFilterWithSED` sign fix are all-negative → all-NaN fits; Part 7c's preflight repairs
  them in place. Delete any stale all-NaN `cigale_runs/<tag>/out/` before re-running —
  `skip_if_done=True` would keep it.